# Symbolic Regression using Next-Generation Transformers

**GSoC 2025 – ML4SCI**

---

## Overview

This notebook implements **SymbolicGPT**, a transformer-based approach to symbolic regression that treats
equation discovery as a language-modelling problem.

### Architecture pipeline

```
Point Cloud  →  T-Net  →  order-invariant embedding
                                 ↓
              GPT Decoder  →  equation skeleton (constants masked as <C>)
                                 ↓
              BFGS  →  constant optimisation  →  final equation
```

### Key references

1. **SymbolicGPT** – Valipour et al., 2021 (arXiv 2106.14131)  
2. **Order-Invariant Embeddings + Sparse Decoding** – Malik et al., NeurIPS ML4PS 2025  
3. **Evolutionary + Transformer Hybrids (PIGP / SymbolicDPO)** – Jha et al., NeurIPS ML4PS 2024  

### Goals

* Learn a generalisable equation generator from thousands of synthetic examples  
* Evaluate on the **AI Feynman** benchmark  
* Report R², RMSE, token accuracy, and exact-match percentage  
* Study robustness to observation noise  

## 1. Install & Import Dependencies

In [ ]:
# ── Install (uncomment in Colab / fresh environment) ─────────────────────────
# !pip install torch sympy scipy numpy matplotlib tqdm requests

import os
import re
import csv
import io
import math
import copy
import time
import tarfile
import pathlib
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import defaultdict

import sympy as sp
from sympy import symbols, lambdify, sympify, latex
from scipy.optimize import minimize
from scipy.stats import pearsonr

try:
    from sklearn.linear_model import LinearRegression
    HAS_SKLEARN = True
except (ImportError, ModuleNotFoundError):
    HAS_SKLEARN = False

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
torch.manual_seed(42)
np.random.seed(42)


## 2. Feynman Dataset Loading & Preprocessing

The **AI Feynman** symbolic regression benchmark (Udrescu & Tegmark, 2020) provides:

| File | Contents |
|---|---|
| `Feynman_with_units.tar.gz` | Numerical data files for Feynman equations (with units) |
| `Feynman_without_units.tar.gz` | Normalised data (dimensionless) |
| `Bonus_with_units.tar.gz` | Bonus equation data (with units) |
| `Bonus_without_units.tar.gz` | Bonus equation data (without units) |
| `FeynmanEquations.csv` | Equation metadata (name, formula, variable count) |
| `FeynmanEquationsDimensionless.csv` | Dimensionless equation metadata |
| `BonusEquations.csv` | Bonus equation metadata |

### Pipeline

1. **Extract** all `.tar.gz` archives in `data/`
2. **Load** equation metadata from CSV files
3. **Map** each equation name → extracted data file path
4. **Build** a unified DataFrame with columns `(name, formula, n_vars, data_path, kind)`

> **Note:** This notebook is configured to use local datasets from `data/` and does not rely on synthetic training data.


In [ ]:
# ── Fast + robust Feynman Dataset Loader ──────────────────────────────────────
import os
import tarfile
import numpy as np
import pandas as pd
import sympy as sp
from pathlib import Path
from collections import defaultdict

# ------------------------------------------------------------------------------
# 1) Find data directory
# ------------------------------------------------------------------------------
def _find_data_dir() -> str:
    cwd = Path.cwd()
    for p in [cwd, *cwd.parents]:
        candidate = p / "data"
        if candidate.is_dir():
            return str(candidate.resolve())
    raise FileNotFoundError("Could not find 'data' directory from current working directory.")

DATA_DIR = _find_data_dir()
print(f"[DataLoader] Using DATA_DIR: {DATA_DIR}")

# ------------------------------------------------------------------------------
# 2) Extract archives only when target folders are missing/empty
# ------------------------------------------------------------------------------
def _folder_has_files(path: str) -> bool:
    return os.path.isdir(path) and len(os.listdir(path)) > 0

def extract_tar_files_if_needed():
    targets = {
        "Feynman_with_units.tar.gz": os.path.join(DATA_DIR, "Feynman_with_units"),
        "Feynman_without_units.tar.gz": os.path.join(DATA_DIR, "Feynman_without_units"),
        "bonus_with_units.tar.gz": os.path.join(DATA_DIR, "bonus_with_units"),
        "bonus_without_units.tar.gz": os.path.join(DATA_DIR, "bonus_without_units"),
    }

    for tar_name, out_dir in targets.items():
        tar_path = os.path.join(DATA_DIR, tar_name)
        if _folder_has_files(out_dir):
            print(f"[DataLoader] Skip extract (already present): {tar_name}")
            continue

        if not os.path.exists(tar_path):
            print(f"[DataLoader] Missing archive: {tar_name}")
            continue

        try:
            print(f"[DataLoader] Extracting {tar_name}...")
            with tarfile.open(tar_path, "r:gz") as tar:
                tar.extractall(DATA_DIR)
        except Exception as exc:
            print(f"[DataLoader] Failed extracting {tar_name}: {exc}")

extract_tar_files_if_needed()

# ------------------------------------------------------------------------------
# 3) Read metadata CSVs
# ------------------------------------------------------------------------------
def _read_csv_or_empty(path: str) -> pd.DataFrame:
    if os.path.exists(path):
        return pd.read_csv(path)
    print(f"[DataLoader] Missing CSV: {path}")
    return pd.DataFrame()

feynman_df = _read_csv_or_empty(os.path.join(DATA_DIR, "FeynmanEquations.csv"))
bonus_df   = _read_csv_or_empty(os.path.join(DATA_DIR, "BonusEquations.csv"))

# ------------------------------------------------------------------------------
# 4) Helpers
# ------------------------------------------------------------------------------
def load_equation_data(file_path: str) -> pd.DataFrame:
    # handles space/tab/comma separated numeric files (including extensionless)
    return pd.read_csv(file_path, sep=r"\s+|,", engine="python", header=None)

def _clean_numeric_table(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.apply(pd.to_numeric, errors="coerce")
          .dropna(axis=1, how="all")
          .dropna(axis=0, how="any")
          .reset_index(drop=True)
    )

def _row_field(row: pd.Series, candidates: list[str], default=""):
    for c in candidates:
        if c in row and pd.notna(row[c]):
            return str(row[c]).strip()
    return default

def _infer_dataset_type(path: str) -> str:
    if not path:
        return "Missing"
    p = path.lower().replace("\\", "/")
    if "/feynman_with_units/" in p:
        return "Feynman_with_units"
    if "/feynman_without_units/" in p:
        return "Feynman_without_units"
    if "/bonus_with_units/" in p:
        return "Bonus_with_units"
    if "/bonus_without_units/" in p:
        return "Bonus_without_units"
    return "Other"

def _normalise_rows(df: pd.DataFrame, kind: str) -> list[dict]:
    rows = []
    if df.empty:
        return rows

    for _, row in df.iterrows():
        name = _row_field(row, ["Filename", "filename", "Name", "name", "Number", "number"])
        # IMPORTANT: use Formula, not Output (Output is target symbol like f/F/E_n)
        formula = _row_field(row, ["Formula", "formula", "Equation", "equation"])
        n_vars_raw = _row_field(row, ["# variables", "#variables", "Num_Vars", "num_vars", "N"])

        n_vars = None
        try:
            n_vars = int(float(n_vars_raw))
        except Exception:
            pass

        if name and formula:
            rows.append({
                "name": name,
                "formula": formula,
                "n_vars": n_vars,
                "kind": kind,
            })
    return rows

def _resolve_data_path(name: str, kind: str, data_dir: str) -> str | None:
    if kind == "feynman":
        candidates = [
            os.path.join(data_dir, "Feynman_with_units", name),
            os.path.join(data_dir, "Feynman_without_units", name),
        ]
    else:
        candidates = [
            os.path.join(data_dir, "bonus_with_units", name),
            os.path.join(data_dir, "bonus_without_units", name),
        ]
    for p in candidates:
        if os.path.exists(p):
            return p
    return None

# ------------------------------------------------------------------------------
# 5) Build catalogue (FAST: map paths only; do not eagerly load all tables)
# ------------------------------------------------------------------------------
def build_dataset_catalogue(data_dir: str) -> list[dict]:
    rows = _normalise_rows(feynman_df, "feynman") + _normalise_rows(bonus_df, "bonus")
    catalogue = []
    for row in rows:
        entry = dict(row)
        entry["data_path"] = _resolve_data_path(entry["name"], entry["kind"], data_dir)
        entry["dataset_type"] = _infer_dataset_type(entry["data_path"])
        catalogue.append(entry)
    return catalogue

CATALOGUE = build_dataset_catalogue(DATA_DIR)

mapped = [e for e in CATALOGUE if e["data_path"]]
print(f"[DataLoader] Equation metadata rows: {len(CATALOGUE)}")
print(f"[DataLoader] Equation↔dataset mapped pairs: {len(mapped)}")

if mapped:
    stats = defaultdict(int)
    for e in mapped:
        stats[e["dataset_type"]] += 1

    print("[DataLoader] Dataset type counts:")
    for k in sorted(stats):
        print(f"  - {k:<22s}: {stats[k]}")

    print("\nSample equation ↔ dataset mappings:")
    for e in mapped[:5]:
        print(f"  {e['name']:<14s} -> {os.path.basename(e['data_path']):<20s} [{e['dataset_type']}]")
else:
    print("[DataLoader] No valid equation↔dataset mappings found.")

# ------------------------------------------------------------------------------
# 6) Lazy point loading API (used by SRDataset later)
# ------------------------------------------------------------------------------
def load_equation_points(entry: dict,
                         n_points: int = 500,
                         rng: np.random.Generator | None = None
                         ) -> tuple[np.ndarray, np.ndarray] | tuple[None, None]:
    path = entry.get("data_path")
    if not path or not os.path.exists(path):
        return None, None

    try:
        df = load_equation_data(path)
        data = _clean_numeric_table(df).to_numpy(dtype=np.float32)
    except Exception:
        return None, None

    if data.ndim != 2 or data.shape[1] < 2:
        return None, None

    n_vars = entry.get("n_vars")
    if n_vars is None:
        n_vars = data.shape[1] - 1
    n_vars = int(n_vars)

    if data.shape[1] < n_vars + 1:
        return None, None

    X = data[:, :n_vars]
    y = data[:, n_vars]

    if n_points is not None and len(X) > n_points:
        if rng is None:
            rng = np.random.default_rng(42)
        idx = rng.choice(len(X), size=n_points, replace=False)
        X, y = X[idx], y[idx]

    return X.astype(np.float32), y.astype(np.float32)

# ------------------------------------------------------------------------------
# 7) Quick sanity check on only 3 mapped files (fast)
# ------------------------------------------------------------------------------
print("\n[DataLoader] Quick file sanity check:")
for e in mapped[:3]:
    try:
        dfi = _clean_numeric_table(load_equation_data(e["data_path"]))
        print(f"  {e['name']:<14s} -> shape={tuple(dfi.shape)}")
    except Exception as exc:
        print(f"  {e['name']:<14s} -> read failed: {exc}")

## 3. Vocabulary & Tokenizer

Equations are represented as token sequences in **prefix (Polish) notation**.

### Why Prefix Notation?

| Property | Infix | Prefix |
|---|---|---|
| Parentheses needed? | Yes | **No** |
| Unambiguous parse? | Requires precedence rules | **Always** |
| Uniform operator arity | No | **Yes** |
| Suitable for LM training? | Messy | **Ideal** |

### Conversion Example

```
sin(x1) + x1²   →   add  sin  x1  pow  x1  <C>
```

### Token Types

| Type | Examples |
|---|---|
| Binary operators | `add sub mul div pow` |
| Unary operators | `sin cos tan exp log sqrt abs tanh` |
| Variables | `x1  x2  x3  x4  x5` |
| Constant placeholder | `<C>` (replaces every numeric literal) |
| Special | `<SOS>  <EOS>  <PAD>  <UNK>` |

Numerical constants in the training targets are replaced by `<C>`, forcing the
language model to learn equation *structure* rather than memorise values.  
A separate BFGS optimisation step (Section 6) recovers the actual constants.


In [ ]:
# ── Prefix-notation vocabulary ────────────────────────────────────────────────

PREFIX_BINARY_OPS = ['add', 'sub', 'mul', 'div', 'pow']
PREFIX_UNARY_OPS  = ['sin', 'cos', 'tan', 'exp', 'log', 'sqrt', 'abs', 'tanh']
PREFIX_OPS        = PREFIX_BINARY_OPS + PREFIX_UNARY_OPS
VARIABLES         = [f'x{i}' for i in range(1, 6)]      # x1 … x5
SPECIAL_TOKENS    = ['<C>', '<SOS>', '<EOS>', '<PAD>', '<UNK>']

ALL_TOKENS  = sorted(set(PREFIX_OPS + VARIABLES + SPECIAL_TOKENS))
VOCAB       = {tok: idx for idx, tok in enumerate(ALL_TOKENS)}
INV_VOCAB   = {idx: tok for tok, idx in VOCAB.items()}
VOCAB_SIZE  = len(VOCAB)

PAD_IDX = VOCAB['<PAD>']
SOS_IDX = VOCAB['<SOS>']
EOS_IDX = VOCAB['<EOS>']
UNK_IDX = VOCAB['<UNK>']

_VARIABLES_SET = set(VARIABLES)

print(f'Vocabulary size : {VOCAB_SIZE}')
print(f'All tokens      : {ALL_TOKENS}')


# ── Sympy-expression → prefix token list ─────────────────────────────────────

def sympy_to_prefix(expr: sp.Expr) -> list:
    """Convert a SymPy expression tree into (possibly nested) prefix tokens."""
    if isinstance(expr, sp.Symbol):
        name = str(expr)
        return [name] if name in _VARIABLES_SET else ['<UNK>']

    if isinstance(expr, sp.Number):
        return ['<C>']

    if isinstance(expr, sp.Add):
        args = list(sp.Add.make_args(expr))
        pos_terms, neg_terms = [], []
        for a in args:
            if a.could_extract_minus_sign():
                neg_terms.append(-a)
            else:
                pos_terms.append(a)

        if pos_terms and neg_terms:
            lhs = sp.Add(*pos_terms) if len(pos_terms) > 1 else pos_terms[0]
            rhs = sp.Add(*neg_terms) if len(neg_terms) > 1 else neg_terms[0]
            return ['sub', sympy_to_prefix(lhs), sympy_to_prefix(rhs)]

        result = sympy_to_prefix(args[0])
        for a in args[1:]:
            result = ['add', result, sympy_to_prefix(a)]
        return result

    if isinstance(expr, sp.Mul):
        args = list(sp.Mul.make_args(expr))
        num_parts, den_parts = [], []
        for a in args:
            if isinstance(a, sp.Pow) and a.exp.is_Number and float(a.exp) < 0:
                den_parts.append(sp.Pow(a.base, -a.exp))
            else:
                num_parts.append(a)

        if den_parts:
            num = sp.Mul(*num_parts) if len(num_parts) > 1 else (num_parts[0] if num_parts else sp.Integer(1))
            den = sp.Mul(*den_parts) if len(den_parts) > 1 else den_parts[0]
            return ['div', sympy_to_prefix(num), sympy_to_prefix(den)]

        result = sympy_to_prefix(args[0])
        for a in args[1:]:
            result = ['mul', result, sympy_to_prefix(a)]
        return result

    if isinstance(expr, sp.Pow):
        if expr.exp == sp.Rational(1, 2):
            return ['sqrt', sympy_to_prefix(expr.base)]
        return ['pow', sympy_to_prefix(expr.base), sympy_to_prefix(expr.exp)]

    unary_map = {
        sp.sin: 'sin', sp.cos: 'cos', sp.tan: 'tan',
        sp.exp: 'exp', sp.log: 'log', sp.sqrt: 'sqrt',
        sp.Abs: 'abs', sp.tanh: 'tanh',
    }
    func = getattr(expr, 'func', None)
    if func in unary_map and len(expr.args) == 1:
        return [unary_map[func], sympy_to_prefix(expr.args[0])]

    return ['<UNK>']


def flatten_prefix(tokens: list | str) -> list[str]:
    """Recursively flatten nested prefix token trees into a 1-D token list."""
    if isinstance(tokens, list):
        result = []
        for t in tokens:
            result.extend(flatten_prefix(t))
        return result
    return [tokens]


def tokenize_equation(eq_str: str) -> list[str]:
    """Parse an equation string and return flattened prefix tokens."""
    expr = sp.sympify(eq_str)
    tokens = sympy_to_prefix(expr)
    return flatten_prefix(tokens)


def equation_str_to_prefix(eq_str: str, n_vars: int = 5) -> list[str]:
    """Parse infix eq string via SymPy, convert to flattened prefix tokens."""
    local_syms = {f'x{i}': sp.Symbol(f'x{i}') for i in range(1, n_vars + 1)}
    try:
        expr = sp.sympify(eq_str, locals=local_syms)
        tokens = sympy_to_prefix(expr)
        return flatten_prefix(tokens)
    except Exception:
        return ['<UNK>']


def tokens_to_ids(tokens: list[str]) -> list[int]:
    return [VOCAB.get(t, UNK_IDX) for t in tokens]


def ids_to_tokens(ids: list[int]) -> list[str]:
    return [INV_VOCAB.get(i, '<UNK>') for i in ids]


def prefix_tokens_to_infix(tokens: list[str]) -> str:
    """
    Reconstruct a human-readable infix expression from prefix tokens.
    (Used for display / debugging.)
    """
    stack = []
    for tok in reversed(tokens):
        if tok in _VARIABLES_SET or tok == '<C>':
            stack.append(tok)
        elif tok in PREFIX_BINARY_OPS:
            a = stack.pop() if stack else '?'
            b = stack.pop() if stack else '?'
            _fmt = {
                'add': f'({a}+{b})',
                'sub': f'({a}-{b})',
                'mul': f'({a}*{b})',
                'div': f'({a}/{b})',
                'pow': f'({a}**{b})',
            }
            stack.append(_fmt.get(tok, f'{tok}({a},{b})'))
        elif tok in PREFIX_UNARY_OPS:
            a = stack.pop() if stack else '?'
            stack.append(f'{tok}({a})')
        # skip <SOS>, <EOS>, <PAD>, <UNK>
    return stack[0] if stack else ''


# ── Quick tests ───────────────────────────────────────────────────────────────
_tokenization_test_cases = [
    ('sin(x1) + x1**2',       'sin x1'),
    ('x1 * x2 - x1 / x2',     'sub mul'),
    ('exp(-x1**2 / 2)',       'exp'),
    ('sqrt(x1**2 + x2**2)',   'sqrt'),
]

print('\nTokenizer smoke tests:')
print(f'  {"Equation":<35s}  Prefix tokens')
print('  ' + '-' * 65)

nested_example = ['add', ['mul', 'x1', 'x2'], ['sin', 'x1']]
print(f"  Flatten nested example: {nested_example} -> {flatten_prefix(nested_example)}")

for eq, expected_substr in _tokenization_test_cases:
    toks = tokenize_equation(eq)
    ok = expected_substr in ' '.join(toks)
    ids = tokens_to_ids(['<SOS>'] + toks + ['<EOS>'])
    back = prefix_tokens_to_infix(toks)
    is_flat = all(not isinstance(t, list) for t in toks)
    status = '✓' if ok and is_flat else '✗'

    tok_str = ' '.join(toks)
    print(f"  {status} {eq:<35s}  {tok_str}")
    print(f'    Flattened: {is_flat}')
    print(f'    IDs: {ids}')
    print(f'    Back to infix: {back}')
    print()

## 4. Real Equation Dataset (Feynman/Bonus)

Training samples are loaded from extracted AI Feynman / Bonus files and converted to prefix token sequences with SymPy.

Each training sample is a tuple `(X_points, y_points, target_token_ids)` where  
- `X_points`, `y_points` form the point cloud for regression  
- `target_token_ids` are the token ids of the masked skeleton equation  


In [ ]:
# ── Equation sampling + dataset (fast, stable, real-data only) ───────────────

UNARY_OPS_SYMPY  = ['sin', 'cos', 'exp', 'log', 'sqrt', 'abs', 'tanh']
BINARY_OPS_SYMPY = ['+', '-', '*', '/', '**']


def safe_eval(expr_str: str, x_vals: dict, eps: float = 1e-8):
    """Evaluate a sympy expression string at given variable values."""
    try:
        syms   = {k: sp.Symbol(k) for k in x_vals}
        expr   = sp.sympify(expr_str, locals=syms)
        func   = sp.lambdify(list(syms.values()), expr, modules='numpy')
        vals   = np.array([x_vals[k] for k in syms], dtype=np.float64)
        result = func(*vals)
        result = np.where(np.isfinite(result), result, np.nan)
        return result
    except Exception:
        return None


def canonicalize_formula_to_x(entry: dict, n_vars: int) -> str:
    """
    Convert metadata formula symbols (theta, sigma, ...) to x1..xN
    so tokenizer can represent variables with your fixed vocabulary.
    """
    try:
        expr = sp.sympify(entry['formula'])
        syms = sorted(list(expr.free_symbols), key=lambda s: str(s))
        syms = syms[:n_vars]
        repl = {s: sp.Symbol(f'x{i+1}') for i, s in enumerate(syms)}
        expr2 = expr.xreplace(repl)
        return str(expr2)
    except Exception:
        return entry.get('formula', '')


class EquationGenerator:
    """Random symbolic equation generator (optional utility)."""

    def __init__(self,
                 max_depth: int = 3,
                 n_vars: int = 2,
                 const_prob: float = 0.5,
                 const_range: tuple = (-2.1, 2.1)):
        self.max_depth   = max_depth
        self.n_vars      = n_vars
        self.const_prob  = const_prob
        self.const_range = const_range
        self.var_names   = [f'x{i+1}' for i in range(n_vars)]

    def _random_expr(self, depth: int) -> sp.Expr:
        if depth >= self.max_depth or (depth > 0 and random.random() < 0.3):
            if random.random() < 0.7:
                return sp.Symbol(random.choice(self.var_names))
            c = random.uniform(*self.const_range)
            return sp.Float(round(c, 2))

        if random.random() < 0.5:  # binary
            op  = random.choice(BINARY_OPS_SYMPY)
            lhs = self._random_expr(depth + 1)
            rhs = self._random_expr(depth + 1)
            op_map = {
                '+': lhs + rhs,
                '-': lhs - rhs,
                '*': lhs * rhs,
                '/': lhs / (rhs + sp.Float(1e-6)),
                '**': lhs ** random.choice([2, 3, 0.5, -1]),
            }
            return op_map[op]

        # unary
        op  = random.choice(UNARY_OPS_SYMPY)
        arg = self._random_expr(depth + 1)
        op_map = {
            'sin': sp.sin, 'cos': sp.cos, 'exp': sp.exp,
            'log': sp.log, 'sqrt': sp.sqrt, 'abs': sp.Abs,
            'tanh': sp.tanh,
        }
        return op_map[op](arg)

    def _add_constants(self, expr: sp.Expr) -> sp.Expr:
        if random.random() < self.const_prob:
            expr = sp.Float(round(random.uniform(*self.const_range), 2)) * expr
        if random.random() < self.const_prob:
            expr = expr + sp.Float(round(random.uniform(*self.const_range), 2))
        return expr

    def generate(self) -> tuple | None:
        try:
            expr = self._add_constants(self._random_expr(0))
            expr = sp.simplify(expr)
            return str(expr), sympy_to_prefix(expr)
        except Exception:
            return None


class PointCloudSampler:
    """Sample point cloud from an equation string."""

    def __init__(self,
                 n_points: int = 50,
                 x_range: tuple = (-3.0, 3.0),
                 n_vars: int = 2,
                 noise_std: float = 0.0):
        self.n_points  = n_points
        self.x_range   = x_range
        self.n_vars    = n_vars
        self.noise_std = noise_std

    def sample(self, expr_str: str) -> tuple:
        var_names = [f'x{i+1}' for i in range(self.n_vars)]
        syms = {v: sp.Symbol(v) for v in var_names}
        try:
            parsed = sp.sympify(expr_str, locals=syms)
            func   = sp.lambdify(list(syms.values()), parsed, modules='numpy')
        except Exception:
            return None, None

        X = np.random.uniform(*self.x_range, size=(self.n_points, self.n_vars)).astype(np.float32)
        try:
            y = func(*[X[:, i] for i in range(self.n_vars)])
            y = np.asarray(y, dtype=np.float32)
            if y.ndim == 0:
                y = np.full((self.n_points,), y, dtype=np.float32)
        except Exception:
            return None, None

        if not np.all(np.isfinite(y)):
            return None, None

        if self.noise_std > 0:
            y = y + np.random.normal(0, self.noise_std * np.std(y) + 1e-8, y.shape).astype(np.float32)
        return X, y


# ── Fast file sampler (reads only small part of huge files) ──────────────────
def load_equation_points(entry: dict,
                         n_points: int = 500,
                         rng: np.random.Generator | None = None
                         ) -> tuple[np.ndarray, np.ndarray] | tuple[None, None]:
    """
    Read only a small chunk from disk for speed.
    """
    path = entry.get("data_path")
    if not path or not os.path.exists(path):
        return None, None

    if rng is None:
        rng = np.random.default_rng(42)

    try:
        read_rows = max(n_points * 6, 512)
        df = pd.read_csv(path, sep=r"\s+|,", engine="python", header=None, nrows=read_rows)
        data = _clean_numeric_table(df).to_numpy(dtype=np.float32)
    except Exception:
        return None, None

    if data.ndim != 2 or data.shape[1] < 2:
        return None, None

    n_vars = entry.get("n_vars")
    if n_vars is None:
        n_vars = data.shape[1] - 1
    n_vars = int(n_vars)

    if data.shape[1] < n_vars + 1:
        return None, None

    X = data[:, :n_vars]
    y = data[:, n_vars]

    if len(X) > n_points:
        idx = rng.choice(len(X), size=n_points, replace=False)
        X, y = X[idx], y[idx]

    return X.astype(np.float32), y.astype(np.float32)


# ── Dataset ───────────────────────────────────────────────────────────────────
class SRDataset(Dataset):
    def __init__(self,
                 size: int = 5000,
                 max_eq_len: int = 80,
                 n_vars: int = 2,
                 n_points: int = 50,
                 noise_std: float = 0.0,
                 max_depth: int = 3,
                 x_range: tuple = (-3.0, 3.0),
                 feynman_catalogue: list | None = None):
        self.max_eq_len = max_eq_len
        self.n_vars = n_vars
        self.data = []

        if not feynman_catalogue:
            raise ValueError("SRDataset requires a non-empty real-data catalogue.")

        # real mapped entries with matching variable count
        eligible = [
            e for e in feynman_catalogue
            if e.get("data_path") and (e.get("n_vars") in (None, n_vars))
        ]
        if not eligible:
            raise ValueError(f"No catalogue entries for n_vars={n_vars} with data files.")

        # precompute tokenizable equations ONCE
        prepared = []
        for e in eligible:
            eq_canon = canonicalize_formula_to_x(e, n_vars=n_vars)
            pfx = equation_str_to_prefix(eq_canon, n_vars=n_vars)
            if "<UNK>" in pfx:
                continue
            if len(pfx) > max_eq_len:
                continue
            prepared.append((e, eq_canon, pfx))

        if not prepared:
            raise RuntimeError(
                f"No tokenizable equations for n_vars={n_vars}. "
                "Try another n_vars or extend tokenizer ops."
            )

        rng = np.random.default_rng(42)
        pbar = tqdm(total=size, desc="Loading real equations")
        attempts = 0
        max_attempts = size * 20

        while len(self.data) < size and attempts < max_attempts:
            attempts += 1
            entry, eq_canon, prefix_toks = prepared[attempts % len(prepared)]

            X, y = load_equation_points(entry, n_points=n_points, rng=rng)
            if X is None or y is None or X.shape[1] != n_vars:
                continue

            if noise_std > 0:
                y = y + np.random.normal(0, noise_std * np.std(y) + 1e-8, y.shape).astype(np.float32)

            self._store(X, y, eq_canon, prefix_toks)
            pbar.update(1)

        pbar.close()
        print(f"[SRDataset] Loaded {len(self.data)} samples (attempts={attempts})")
        if len(self.data) == 0:
            raise RuntimeError("Loaded 0 samples. Check tokenizer coverage / n_vars.")

    def _store(self, X, y, eq_str, prefix_toks):
        tokens = ["<SOS>"] + prefix_toks + ["<EOS>"]
        if len(tokens) > self.max_eq_len + 2:
            return
        if any(t == "<UNK>" for t in prefix_toks):
            return
        ids = tokens_to_ids(tokens)
        cloud = np.concatenate([X, y.reshape(-1, 1)], axis=1).astype(np.float32)
        self.data.append({
            "cloud": cloud,
            "ids": ids,
            "eq_str": eq_str,
            "prefix": " ".join(prefix_toks),
        })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        ids = item["ids"]
        input_ids = torch.tensor(ids[:-1], dtype=torch.long)
        labels = torch.tensor(ids[1:], dtype=torch.long)
        cloud = torch.tensor(item["cloud"], dtype=torch.float32)
        return cloud, input_ids, labels, item["eq_str"], item["prefix"]


def collate_fn(batch):
    clouds, input_ids_list, labels_list, eq_strs, prefixes = zip(*batch)

    max_len = max(x.size(0) for x in input_ids_list)
    input_padded = torch.stack([
        F.pad(x, (0, max_len - x.size(0)), value=PAD_IDX) for x in input_ids_list
    ])
    labels_padded = torch.stack([
        F.pad(x, (0, max_len - x.size(0)), value=PAD_IDX) for x in labels_list
    ])

    max_pts = max(c.size(0) for c in clouds)
    clouds_padded = torch.stack([
        F.pad(c, (0, 0, 0, max_pts - c.size(0)), value=0.0) for c in clouds
    ])
    return clouds_padded, input_padded, labels_padded, list(eq_strs), list(prefixes)


# ── Smoke test ────────────────────────────────────────────────────────────────
print("\nGenerating smoke-test dataset (real equations)...")
if not CATALOGUE:
    raise RuntimeError("CATALOGUE is empty. Ensure data loading cell ran successfully.")

smoke_ds = SRDataset(
    size=min(20, len(CATALOGUE)),
    n_vars=2,
    n_points=20,
    max_depth=2,
    feynman_catalogue=CATALOGUE,
)

smoke_dl = DataLoader(smoke_ds, batch_size=8, collate_fn=collate_fn)
clouds, inp, lbl, eqs, pfx = next(iter(smoke_dl))

print(f"Cloud shape  : {clouds.shape}")
print(f"Input shape  : {inp.shape}")
print(f"Label shape  : {lbl.shape}")
print(f"Sample eq    : {eqs[0]}")
print(f"Sample prefix: {pfx[0]}")


# ── Quick verification ────────────────────────────────────────────────────────
def verify_equation(eq_str, data):
    try:
        expr = sp.sympify(eq_str)
    except Exception:
        return None

    variables = sorted(expr.free_symbols, key=lambda s: str(s))
    if len(variables) == 0:
        return None

    X = data.iloc[:, :-1].values
    y_true = data.iloc[:, -1].values
    if X.shape[1] < len(variables):
        return None

    f = sp.lambdify(variables, expr, "numpy")
    try:
        y_pred = f(*[X[:, i] for i in range(len(variables))])
        y_pred = np.asarray(y_pred, dtype=np.float64)
        if y_pred.ndim == 0:
            y_pred = np.full_like(y_true, y_pred, dtype=np.float64)
        if y_pred.shape[0] != y_true.shape[0]:
            return None
        if not np.all(np.isfinite(y_pred)):
            return None
        return float(np.mean((y_true - y_pred) ** 2))
    except Exception:
        return None

verify_sample = smoke_ds.data[0]
verify_eq = verify_sample["eq_str"]
verify_cloud = verify_sample["cloud"]
verify_cols = [f"x{i+1}" for i in range(verify_cloud.shape[1] - 1)] + ["y"]
verify_data = pd.DataFrame(verify_cloud, columns=verify_cols)

print("Equation:", verify_eq)
print("Sample data:\n", verify_data.head())
print("Verification error:", verify_equation(verify_eq, verify_data))

In [ ]:
# ── SRDataset (Augmentation-enabled override) ─────────────────────────────────
class SRDataset(Dataset):
    """Symbolic-regression dataset with optional data augmentation."""

    def __init__(self,
                 size: int = 5000,
                 max_eq_len: int = 80,
                 n_vars: int = 2,
                 n_points: int = 50,
                 noise_std: float = 0.0,
                 max_depth: int = 3,
                 x_range: tuple = (-3.0, 3.0),
                 feynman_catalogue: list | None = None,
                 enable_augmentation: bool = True,
                 aug_prob_var_permute: float = 0.3,
                 aug_prob_noise: float = 0.3,
                 aug_prob_subsample: float = 0.2,
                 aug_prob_const_scale: float = 0.2,
                 aug_prob_const_shift: float = 0.2):
        self.max_eq_len = max_eq_len
        self.n_vars = n_vars
        self.data = []

        self.enable_augmentation = enable_augmentation
        self.aug_prob_var_permute = aug_prob_var_permute
        self.aug_prob_noise = aug_prob_noise
        self.aug_prob_subsample = aug_prob_subsample
        self.aug_prob_const_scale = aug_prob_const_scale
        self.aug_prob_const_shift = aug_prob_const_shift

        if not feynman_catalogue:
            raise ValueError('SRDataset requires a non-empty real-data catalogue.')

        eligible = [
            e for e in feynman_catalogue
            if e.get('data_path') and (e.get('n_vars') in (None, n_vars))
        ]
        if not eligible:
            raise ValueError(f'No catalogue entries found for n_vars={n_vars} with data files.')

        rng = np.random.default_rng(42)
        order = rng.permutation(len(eligible)).tolist()

        attempts = 0
        idx = 0
        pbar = tqdm(total=size, desc='Loading real equations (augmented)')
        while len(self.data) < size and attempts < max(size * 20, len(eligible) * 5):
            attempts += 1
            entry = eligible[order[idx % len(order)]]
            idx += 1

            X, y = load_equation_points(entry, n_points=n_points, rng=rng)
            if X is None:
                continue

            if noise_std > 0:
                y = y + np.random.normal(0, noise_std * np.std(y) + 1e-8, y.shape)

            prefix_toks = equation_str_to_prefix(entry['formula'], n_vars=n_vars)
            before = len(self.data)
            self._store(X, y, entry['formula'], prefix_toks, n_vars=n_vars)
            if len(self.data) > before:
                pbar.update(1)

        pbar.close()
        print(f'[SRDataset-Aug] Loaded {len(self.data)} real samples (attempts={attempts})')

    def augment_variable_permutation(self, X, equation_str, n_vars):
        if n_vars <= 1:
            return X, equation_str
        perm = np.random.permutation(n_vars)
        X_perm = X[:, perm]
        try:
            syms = {f'x{i+1}': sp.Symbol(f'x{i+1}') for i in range(n_vars)}
            expr = sp.sympify(equation_str, locals=syms)
            repl = {syms[f'x{i+1}']: syms[f'x{perm[i]+1}'] for i in range(n_vars)}
            eq_perm = str(sp.simplify(expr.xreplace(repl)))
        except Exception:
            eq_perm = equation_str
        return X_perm, eq_perm

    def augment_gaussian_noise(self, y, noise_level=0.02):
        if np.std(y) < 1e-8:
            return y
        noise = np.random.normal(0, noise_level * np.std(y), y.shape)
        return y + noise

    def augment_point_subsample(self, X, y, sample_ratio=0.85):
        n_samples = int(len(X) * sample_ratio)
        n_samples = max(min(n_samples, len(X)), min(20, len(X)))
        if n_samples >= len(X):
            return X, y
        idx = np.random.choice(len(X), size=n_samples, replace=False)
        return X[idx], y[idx]

    def augment_constant_scaling(self, y, equation_str):
        scale = float(np.random.uniform(0.5, 5.0))
        y_scaled = y * scale
        try:
            expr = sp.sympify(equation_str)
            eq_scaled = str(sp.simplify(scale * expr))
        except Exception:
            eq_scaled = f'{scale:.4f} * ({equation_str})'
        return y_scaled, eq_scaled

    def augment_constant_shift(self, y, equation_str):
        offset = float(np.random.uniform(-1.0, 1.0))
        y_shifted = y + offset
        try:
            expr = sp.sympify(equation_str)
            eq_shifted = str(sp.simplify(expr + offset))
        except Exception:
            if offset >= 0:
                eq_shifted = f'({equation_str}) + {offset:.4f}'
            else:
                eq_shifted = f'({equation_str}) - {abs(offset):.4f}'
        return y_shifted, eq_shifted

    def apply_augmentations(self, X, y, equation_str, n_vars):
        if not self.enable_augmentation:
            return X, y, equation_str

        X_aug, y_aug, eq_aug = X.copy(), y.copy(), equation_str

        if np.random.random() < self.aug_prob_var_permute:
            X_aug, eq_aug = self.augment_variable_permutation(X_aug, eq_aug, n_vars)

        if np.random.random() < self.aug_prob_noise:
            y_aug = self.augment_gaussian_noise(y_aug, noise_level=0.02)

        if np.random.random() < self.aug_prob_subsample:
            ratio = float(np.random.uniform(0.8, 1.0))
            X_aug, y_aug = self.augment_point_subsample(X_aug, y_aug, sample_ratio=ratio)

        if np.random.random() < self.aug_prob_const_scale:
            y_aug, eq_aug = self.augment_constant_scaling(y_aug, eq_aug)

        if np.random.random() < self.aug_prob_const_shift:
            y_aug, eq_aug = self.augment_constant_shift(y_aug, eq_aug)

        return X_aug, y_aug, eq_aug

    def _store(self, X, y, eq_str, prefix_toks, n_vars):
        X_aug, y_aug, eq_aug = self.apply_augmentations(X, y, eq_str, n_vars)

        if eq_aug != eq_str:
            try:
                prefix_toks = equation_str_to_prefix(eq_aug, n_vars=n_vars)
            except Exception:
                X_aug, y_aug, eq_aug = X, y, eq_str

        tokens = ['<SOS>'] + prefix_toks + ['<EOS>']
        if len(tokens) > self.max_eq_len + 2:
            return
        if any(t == '<UNK>' for t in prefix_toks):
            return

        ids = tokens_to_ids(tokens)
        cloud = np.concatenate([X_aug, y_aug.reshape(-1, 1)], axis=1).astype(np.float32)

        self.data.append({
            'cloud': cloud,
            'ids': ids,
            'eq_str': eq_aug,
            'prefix': ' '.join(prefix_toks),
            'prefix_str': ' '.join(prefix_toks),
        })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        ids = item['ids']
        input_ids = torch.tensor(ids[:-1], dtype=torch.long)
        labels = torch.tensor(ids[1:], dtype=torch.long)
        cloud = torch.tensor(item['cloud'], dtype=torch.float32)
        return cloud, input_ids, labels, item['eq_str'], item['prefix']

In [ ]:
print("\n" + "="*80)
print("TESTING DATA AUGMENTATION")
print("="*80 + "\n")

# ---------------------------------------------------------------------
# Build a valid, tokenizable 2-variable mini-catalogue for this test
# ---------------------------------------------------------------------
def _canon_formula(eq: str, n_vars: int = 2) -> str:
    """
    Canonicalize variable names to x1..xN so tokenizer vocabulary can parse.
    """
    try:
        expr = sp.sympify(eq)
        syms = sorted(list(expr.free_symbols), key=lambda s: str(s))[:n_vars]
        repl = {s: sp.Symbol(f'x{i+1}') for i, s in enumerate(syms)}
        return str(expr.xreplace(repl))
    except Exception:
        return eq

base_catalogue = train_catalogue if 'train_catalogue' in globals() else CATALOGUE

candidate_catalogue = []
for e in base_catalogue:
    if not e.get('data_path'):
        continue
    if e.get('n_vars') not in (None, 2):
        continue

    e2 = dict(e)
    e2['formula'] = _canon_formula(e2['formula'], n_vars=2)

    pfx = equation_str_to_prefix(e2['formula'], n_vars=2)
    if '<UNK>' in pfx:
        continue

    candidate_catalogue.append(e2)
    if len(candidate_catalogue) >= 15:
        break

if len(candidate_catalogue) < 3:
    raise RuntimeError("Not enough valid 2-variable equations found for augmentation test.")

print(f"Using {len(candidate_catalogue)} valid equations for augmentation test.\n")

# ---------------------------------------------------------------------
# Create augmentation test datasets (small + fast settings)
# ---------------------------------------------------------------------
test_aug_ds = SRDataset(
    size=20,
    n_vars=2,
    n_points=20,
    feynman_catalogue=candidate_catalogue,
    enable_augmentation=True,
    aug_prob_var_permute=1.0,   # force visible augmentation for demo
    aug_prob_noise=0.3,
    aug_prob_subsample=0.2,
    aug_prob_const_scale=0.2,
    aug_prob_const_shift=0.2,
)

print("Sample augmented equations:\n")
for i in range(min(5, len(test_aug_ds))):
    sample = test_aug_ds.data[i]
    print(f"Sample {i+1}:")
    print(f"  Equation: {sample['eq_str']}")
    print(f"  Prefix:   {sample.get('prefix_str', sample.get('prefix', ''))}")
    print(f"  Cloud shape: {sample['cloud'].shape}")
    print()

ds_no_aug = SRDataset(
    size=20,
    n_vars=2,
    n_points=20,
    feynman_catalogue=candidate_catalogue,
    enable_augmentation=False,
)

ds_with_aug = SRDataset(
    size=20,
    n_vars=2,
    n_points=20,
    feynman_catalogue=candidate_catalogue,
    enable_augmentation=True,
    aug_prob_var_permute=0.5,
    aug_prob_noise=0.3,
    aug_prob_subsample=0.2,
    aug_prob_const_scale=0.2,
    aug_prob_const_shift=0.2,
)

print(f"\nDataset without augmentation: {len(ds_no_aug)} samples")
print(f"Dataset with augmentation: {len(ds_with_aug)} samples")

unique_no_aug = len(set(s['eq_str'] for s in ds_no_aug.data))
unique_with_aug = len(set(s['eq_str'] for s in ds_with_aug.data))

print(f"\nUnique equations without augmentation: {unique_no_aug}")
print(f"Unique equations with augmentation: {unique_with_aug}")
print(f"Diversity increase: +{unique_with_aug - unique_no_aug} unique variants")

print("\n✅ Data augmentation test complete!")

## 5. Model Architecture

### 5.1 T-Net – Order-Invariant Point Cloud Encoder

Inspired by PointNet (Qi et al., 2017) and re-interpreted as a **semantic tree encoder** in  
Malik et al. (NeurIPS ML4PS 2025), the T-Net maps a variable-size point cloud  
`X ∈ ℝ^{N×(d+1)}` to a fixed-size embedding `w ∈ ℝ^e` through:

1. Per-point shared MLP layers → `N × 4e`
2. Global max-pooling → `1 × 4e`  (order-invariant)
3. Two FC layers → `1 × e`

In [ ]:
class TNet(nn.Module):
    """
    Order-invariant point-cloud encoder.
    Maps [B, N, d+1] → [B, emb_dim] and supports variable-length point clouds.
    Uses LayerNorm to normalise each point across its feature dimension, which
    avoids dependence on batch-level statistics.
    """

    def __init__(self, in_dim: int, emb_dim: int = 256):
        super().__init__()
        self.input_norm = nn.LayerNorm(in_dim)

        # Shared MLP applied independently to each point
        self.shared_mlp = nn.Sequential(
            nn.Linear(in_dim, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, emb_dim * 2),
            nn.ReLU(),
            nn.Linear(emb_dim * 2, emb_dim * 4),
            nn.ReLU(),
        )

        # Aggregation MLP
        self.fc1 = nn.Sequential(
            nn.Linear(emb_dim * 4, emb_dim * 2),
            nn.ReLU(),
        )
        self.fc2 = nn.Linear(emb_dim * 2, emb_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, N, d+1]
        x = self.input_norm(x)
        x = self.shared_mlp(x)
        x = torch.max(x, dim=1)[0]

        x = self.fc1(x)
        x = self.fc2(x)
        return x


# Smoke test
tnet   = TNet(in_dim=3, emb_dim=64).to(DEVICE)
x_test = torch.randn(4, 50, 3).to(DEVICE)
emb    = tnet(x_test)
print(f'TNet output: {emb.shape}')   # should be [4, 64]


### 5.2 Transformer Decoder (GPT-style)

A causal (prefix-self-attention) Transformer decoder generates the equation skeleton
token by token.  The point-cloud embedding `w_D` is injected by adding it  
(broadcast) to every position of the token-embedding + positional-encoding matrix,  
following the SymbolicGPT formulation:

```
h = W_p + W_D + X_eq * W_t
```

where `W_D` is `w_D` expanded to match sequence length.

In [ ]:
def generate_causal_mask(seq_len: int) -> torch.Tensor:
    """Create an upper-triangular boolean mask that blocks future positions."""
    return torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()


class CausalSelfAttention(nn.Module):
    """Multi-head causal (masked) self-attention block."""

    def __init__(self, emb_dim: int, n_heads: int, dropout: float = 0.1,
                 max_len: int = 256):
        super().__init__()
        assert emb_dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = emb_dim // n_heads

        self.qkv_proj  = nn.Linear(emb_dim, 3 * emb_dim, bias=False)
        self.out_proj  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.register_buffer('causal_mask', generate_causal_mask(max_len), persistent=False)

    def forward(self, x: torch.Tensor,
                key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        B, T, C = x.shape
        H, D    = self.n_heads, self.head_dim

        qkv = self.qkv_proj(x)
        q, k, v = qkv.split(C, dim=-1)

        q = q.view(B, T, H, D).transpose(1, 2)
        k = k.view(B, T, H, D).transpose(1, 2)
        v = v.view(B, T, H, D).transpose(1, 2)

        scale  = D ** -0.5
        scores = (q @ k.transpose(-2, -1)) * scale

        causal_mask = self.causal_mask[:T, :T]
        scores = scores.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))

        if key_padding_mask is not None:
            scores = scores.masked_fill(
                key_padding_mask.unsqueeze(1).unsqueeze(2), float('-inf'))

        attn = F.softmax(scores, dim=-1)
        attn = self.attn_drop(attn)

        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)


class TransformerBlock(nn.Module):
    """Single GPT-style transformer block."""

    def __init__(self, emb_dim: int, n_heads: int,
                 ff_mult: int = 4, dropout: float = 0.1, max_len: int = 256):
        super().__init__()
        self.ln1   = nn.LayerNorm(emb_dim)
        self.attn  = CausalSelfAttention(emb_dim, n_heads, dropout, max_len)
        self.ln2   = nn.LayerNorm(emb_dim)
        self.ff    = nn.Sequential(
            nn.Linear(emb_dim, ff_mult * emb_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_mult * emb_dim, emb_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x, key_padding_mask=None):
        x = x + self.attn(self.ln1(x), key_padding_mask)
        x = x + self.ff(self.ln2(x))
        return x


class SymbolicGPT(nn.Module):
    """
    Full SymbolicGPT model:
        T-Net encoder  +  GPT decoder
    """

    def __init__(self,
                 in_dim: int,
                 vocab_size: int,
                 emb_dim: int = 256,
                 n_heads: int = 8,
                 n_layers: int = 4,
                 max_seq_len: int = 100,
                 dropout: float = 0.1):
        super().__init__()
        self.emb_dim = emb_dim

        self.tnet = TNet(in_dim=in_dim, emb_dim=emb_dim)

        self.tok_emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        # Project dataset embedding into decoder token embedding space before fusion.
        self.dataset_proj = nn.Linear(emb_dim, emb_dim)
        self.pos_emb = nn.Embedding(max_seq_len, emb_dim)
        self.emb_drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(emb_dim, n_heads, dropout=dropout, max_len=max_seq_len)
            for _ in range(n_layers)
        ])
        self.ln_f    = nn.LayerNorm(emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    def forward(self,
                cloud: torch.Tensor,
                input_ids: torch.Tensor,
                key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        B, T = input_ids.shape

        dataset_embedding = self.dataset_proj(self.tnet(cloud))

        positions = torch.arange(T, device=input_ids.device).unsqueeze(0)
        token_embedding = self.tok_emb(input_ids)
        positional_embedding = self.pos_emb(positions)

        combined_embedding = (
            token_embedding
            + dataset_embedding.unsqueeze(1)
            + positional_embedding
        )
        h = self.emb_drop(combined_embedding)

        for block in self.blocks:
            h = block(h, key_padding_mask)

        h = self.ln_f(h)
        return self.lm_head(h)

    @torch.no_grad()
    def generate(self,
                 cloud: torch.Tensor,
                 max_new_tokens: int = 80,
                 temperature: float = 0.8,
                 top_k: int = 40) -> list:
        self.eval()
        device   = next(self.parameters()).device
        ids      = torch.tensor([[SOS_IDX]], dtype=torch.long, device=device)
        generated = []

        for _ in range(max_new_tokens):
            logits = self(cloud, ids)
            logits = logits[:, -1, :] / temperature

            if top_k > 0:
                vals, _ = torch.topk(logits, top_k)
                threshold = vals[:, -1].unsqueeze(-1)
                logits = logits.masked_fill(logits < threshold, float('-inf'))

            probs  = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            ids     = torch.cat([ids, next_id], dim=1)
            generated.append(next_id.item())

            if next_id.item() == EOS_IDX:
                break

        return generated


# ── Smoke test ────────────────────────────────────────────────────────────────
N_VARS = 2
model  = SymbolicGPT(
    in_dim      = N_VARS + 1,
    vocab_size  = VOCAB_SIZE,
    emb_dim     = 128,
    n_heads     = 4,
    n_layers    = 2,
    max_seq_len = 100,
    dropout     = 0.1,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params:,}')

cloud_t  = torch.randn(4, 50, N_VARS + 1).to(DEVICE)
inp_t    = torch.randint(0, VOCAB_SIZE, (4, 20)).to(DEVICE)
logits_t = model(cloud_t, inp_t)
print(f'Logits shape: {logits_t.shape}')


## 6. Training

We train SymbolicGPT with **cross-entropy loss** (ignoring `<PAD>` tokens)  
using the **Adam** optimiser with cosine learning-rate decay.

In [ ]:
# ── Hyper-parameters ─────────────────────────────────────────────────────────
CFG = dict(
    # Dataset
    train_size   = 8000,
    val_size     = 1000,
    n_vars       = 2,
    n_points     = 50,
    max_depth    = 3,
    # Model
    emb_dim      = 256,
    n_heads      = 8,
    n_layers     = 4,
    max_seq_len  = 100,
    dropout      = 0.1,
    # Training
    epochs       = 10,
    batch_size   = 64,
    lr           = 3e-4,
    weight_decay = 1e-4,
    grad_clip    = 1.0,
)

print('Configuration:', CFG)

In [ ]:
# ── Build datasets (fast + robust) ────────────────────────────────────────────
if not CATALOGUE:
    raise RuntimeError("No local equation catalogue found. Populate data/ before training.")

# Use full configured values (caps removed)
CFG['train_size'] = CFG.get('train_size', 8000)
CFG['val_size']   = CFG.get('val_size', 1000)
CFG['batch_size'] = CFG.get('batch_size', 64)
CFG['n_points']   = CFG.get('n_points', 50)

def _canon_formula(eq: str, n_vars: int) -> str:
    try:
        expr = sp.sympify(eq)
        syms = sorted(list(expr.free_symbols), key=lambda s: str(s))[:n_vars]
        repl = {s: sp.Symbol(f'x{i+1}') for i, s in enumerate(syms)}
        return str(expr.xreplace(repl))
    except Exception:
        return eq

def _is_tokenizable_for_nvars(entry: dict, n_vars: int) -> bool:
    try:
        eq = _canon_formula(entry['formula'], n_vars=n_vars)
        pfx = equation_str_to_prefix(eq, n_vars=n_vars)
        return ('<UNK>' not in pfx) and (len(pfx) <= 80)
    except Exception:
        return False

# keep real files + matching n_vars + tokenizable
usable_catalogue = []
for e in CATALOGUE:
    if not e.get('data_path'):
        continue
    if e.get('n_vars') not in (None, CFG['n_vars']):
        continue
    if not _is_tokenizable_for_nvars(e, CFG['n_vars']):
        continue
    e2 = dict(e)
    e2['formula'] = _canon_formula(e2['formula'], CFG['n_vars'])  # important
    usable_catalogue.append(e2)

if len(usable_catalogue) < 10:
    raise RuntimeError(
        f"Too few usable equations ({len(usable_catalogue)}). "
        f"Try CFG['n_vars']=1 or extend tokenizer ops."
    )

rng = np.random.default_rng(42)
perm = rng.permutation(len(usable_catalogue))
split = max(1, int(0.8 * len(usable_catalogue)))
train_catalogue = [usable_catalogue[i] for i in perm[:split]]
val_catalogue   = [usable_catalogue[i] for i in perm[split:]] or train_catalogue

print(f"Usable equations: {len(usable_catalogue)} | Train: {len(train_catalogue)} | Val: {len(val_catalogue)}")
print(f"Using sizes -> train_size={CFG['train_size']}, val_size={CFG['val_size']}, n_points={CFG['n_points']}")

print("Building training set (with augmentation)...")
train_ds = SRDataset(
    size=CFG['train_size'],
    n_vars=CFG['n_vars'],
    n_points=CFG['n_points'],
    max_depth=CFG['max_depth'],
    feynman_catalogue=train_catalogue,
    enable_augmentation=True,
    aug_prob_var_permute=0.3,
    aug_prob_noise=0.3,
    aug_prob_subsample=0.2,
    aug_prob_const_scale=0.2,
    aug_prob_const_shift=0.2,
)

print("Building validation set (without augmentation)...")
val_ds = SRDataset(
    size=CFG['val_size'],
    n_vars=CFG['n_vars'],
    n_points=CFG['n_points'],
    max_depth=CFG['max_depth'],
    feynman_catalogue=val_catalogue,
    enable_augmentation=False,
)

train_dl = DataLoader(
    train_ds,
    batch_size=CFG['batch_size'],
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0
)
val_dl = DataLoader(
    val_ds,
    batch_size=CFG['batch_size'],
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")
print(f"Train batches: {len(train_dl)} | Val batches: {len(val_dl)}")

In [ ]:
# ── Instantiate model & optimiser ─────────────────────────────────────────────
MAX_VARS = 5

model = SymbolicGPT(
    in_dim      = MAX_VARS + 1,
    vocab_size  = VOCAB_SIZE,
    emb_dim     = CFG['emb_dim'],
    n_heads     = CFG['n_heads'],
    n_layers    = CFG['n_layers'],
    max_seq_len = CFG['max_seq_len'],
    dropout     = CFG['dropout'],
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG['lr'],
    weight_decay=CFG['weight_decay'],
)

total_steps = CFG['epochs'] * len(train_dl)
scheduler   = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=max(total_steps, 1), eta_min=1e-6
)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

print(f'Model params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print(f'Model input dimension supports up to {MAX_VARS} variables (+1 target).')

In [ ]:
sample_clouds, _, _, _, _ = next(iter(train_dl))
in_dim_runtime = int(sample_clouds.shape[-1])

print(f"[Init] Runtime cloud last-dim = {in_dim_runtime}")
print(f"[Init] CFG n_vars + 1         = {CFG['n_vars'] + 1}")

model = SymbolicGPT(
    in_dim=in_dim_runtime,
    vocab_size=VOCAB_SIZE,
    emb_dim=CFG['emb_dim'],
    n_heads=CFG['n_heads'],
    n_layers=CFG['n_layers'],
    max_seq_len=CFG['max_seq_len'],
    dropout=CFG['dropout'],
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG['lr'],
    weight_decay=CFG['weight_decay'],
)

total_steps = max(CFG['epochs'] * len(train_dl), 1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=total_steps,
    eta_min=1e-6,
)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[Init] Model params: {n_params:,}")
print(f"[Init] Model in_dim : {in_dim_runtime}")

In [ ]:
# ── Curriculum learning for multi-variable training (2 → 3 → 4/5) ───────────
def load_equation_points_padded(entry: dict,
                                n_points: int = 500,
                                max_vars: int = 5,
                                rng: np.random.Generator | None = None) -> tuple:
    """Load equation points and pad/truncate feature dimensions to max_vars."""
    X_orig, y = load_equation_points(entry, n_points=n_points, rng=rng)
    if X_orig is None:
        return None, None, None

    actual_n_vars = int(X_orig.shape[1])

    if actual_n_vars < max_vars:
        pad = np.zeros((X_orig.shape[0], max_vars - actual_n_vars), dtype=np.float32)
        X_padded = np.concatenate([X_orig, pad], axis=1)
    else:
        X_padded = X_orig[:, :max_vars]

    return X_padded.astype(np.float32), y.astype(np.float32), actual_n_vars


class CurriculumTrainer:
    """Progressive training on equations with increasing variable counts."""

    def __init__(self, model, catalogue, device, max_vars=5):
        self.model = model
        self.device = device
        self.max_vars = max_vars

        self.equations_by_nvars = {}
        for n_vars in [1, 2, 3, 4, 5]:
            if n_vars == 4:
                self.equations_by_nvars[n_vars] = [
                    e for e in catalogue
                    if e.get('n_vars') in (4, 5) and e.get('data_path')
                ]
            elif n_vars == 5:
                self.equations_by_nvars[n_vars] = []
            else:
                self.equations_by_nvars[n_vars] = [
                    e for e in catalogue
                    if e.get('n_vars') == n_vars and e.get('data_path')
                ]

        print('Curriculum dataset distribution:')
        for n_vars in [1, 2, 3, 4]:
            print(f'  {n_vars}-variable equations: {len(self.equations_by_nvars.get(n_vars, []))}')

    def train_stage(self, n_vars, epochs, lr, train_size, batch_size=64):
        print(f"\n{'='*80}")
        print(f'STAGE: Training on {n_vars}-variable equations')
        print(f'Epochs: {epochs}, LR: {lr}, Dataset size: {train_size}')
        print(f"{'='*80}\n")

        stage_catalogue = self.equations_by_nvars.get(n_vars, [])
        if not stage_catalogue:
            print(f'⚠️  No equations found for {n_vars} variables, skipping stage')
            return {'train_loss': [], 'train_acc': []}

        train_ds = SRDataset(
            size=train_size,
            n_vars=n_vars,
            n_points=CFG.get('n_points', 50),
            max_depth=CFG.get('max_depth', 3),
            feynman_catalogue=stage_catalogue,
            enable_augmentation=True,
            aug_prob_var_permute=0.3,
            aug_prob_noise=0.3,
            aug_prob_subsample=0.2,
            aug_prob_const_scale=0.2,
            aug_prob_const_shift=0.2,
        )

        train_dl = DataLoader(
            train_ds,
            batch_size=batch_size,
            shuffle=True,
            collate_fn=collate_fn,
            num_workers=0,
        )

        optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=lr,
            weight_decay=1e-4,
        )

        stage_steps = max(epochs * max(len(train_dl), 1), 1)
        stage_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=stage_steps,
            eta_min=lr / 10,
        )

        global scheduler
        scheduler = stage_scheduler

        history = {'train_loss': [], 'train_acc': []}
        for epoch in range(1, epochs + 1):
            train_loss, train_acc = run_epoch(
                self.model,
                train_dl,
                optimizer=optimizer,
                grad_clip=CFG.get('grad_clip', 1.0),
                desc=f'Stage {n_vars}-var, Epoch {epoch}/{epochs}',
            )
            history['train_loss'].append(float(train_loss))
            history['train_acc'].append(float(train_acc))
            print(f'  Epoch {epoch:2d}: loss={train_loss:.4f}, acc={train_acc:.4f}')

        return history

    def evaluate_generalization(self, stage_name):
        print(f"\n{'─'*80}")
        print(f'EVALUATION: {stage_name}')
        print(f"{'─'*80}")

        if 'evaluate_model' not in globals():
            print('  evaluate_model() not available yet, skipping this stage evaluation.')
            return {}

        results = {}
        for n_vars in [1, 2, 3, 4]:
            eqs = self.equations_by_nvars.get(n_vars, [])
            if not eqs:
                continue

            val_ds = SRDataset(
                size=max(20, min(200, len(eqs) * 10)),
                n_vars=n_vars,
                n_points=CFG.get('n_points', 50),
                max_depth=CFG.get('max_depth', 3),
                feynman_catalogue=eqs,
                enable_augmentation=False,
            )

            if len(val_ds) == 0:
                continue

            metrics = evaluate_model(
                self.model,
                val_ds,
                n_samples=min(100, len(val_ds)),
                n_vars=n_vars,
            )
            results[n_vars] = metrics

            print(
                f"  {n_vars}-var: R²={metrics['r2_mean']:.4f}, "
                f"RMSE={metrics['rmse_mean']:.4f}, "
                f"TokAcc={metrics['token_acc_mean']*100:.2f}%, "
                f"Exact={metrics['exact_match']*100:.2f}%"
            )

        return results

    def train_curriculum(self, stages_config):
        full_history = {'stages': [], 'evaluations': []}

        print('\n' + '='*80)
        print('STARTING CURRICULUM TRAINING')
        print('='*80)

        for stage_idx, stage in enumerate(stages_config):
            stage_history = self.train_stage(
                n_vars=stage['n_vars'],
                epochs=stage['epochs'],
                lr=stage['lr'],
                train_size=stage['train_size'],
                batch_size=stage.get('batch_size', 64),
            )

            full_history['stages'].append({
                'n_vars': stage['n_vars'],
                'history': stage_history,
            })

            eval_results = self.evaluate_generalization(
                f"After Stage {stage_idx + 1} ({stage['n_vars']}-var training)"
            )
            full_history['evaluations'].append({
                'after_stage': stage_idx + 1,
                'n_vars': stage['n_vars'],
                'results': eval_results,
            })

        print('\n' + '='*80)
        print('CURRICULUM TRAINING COMPLETE')
        print('='*80)

        return full_history

In [ ]:
# ── Training loop (fresh run) ────────────────────────────────────────────────
def token_accuracy(logits, labels, pad_idx=PAD_IDX):
    preds = logits.argmax(-1)
    mask = labels != pad_idx
    correct = (preds == labels) & mask
    return correct.sum().item() / max(mask.sum().item(), 1)

def run_epoch(model, loader, optimizer=None, grad_clip=None, desc='Train'):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss, total_acc, n_batches = 0.0, 0.0, 0
    pbar = tqdm(loader, desc=desc, leave=False)

    for clouds, inp, lbl, _, _ in pbar:
        clouds = clouds.to(DEVICE)
        inp = inp.to(DEVICE)
        lbl = lbl.to(DEVICE)
        pad_mask = (inp == PAD_IDX)

        with torch.set_grad_enabled(is_train):
            logits = model(clouds, inp, key_padding_mask=pad_mask)
            B, T, V = logits.shape
            loss = criterion(logits.reshape(B * T, V), lbl.reshape(B * T))
            acc = token_accuracy(logits, lbl)

        if is_train:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            if grad_clip is not None:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            scheduler.step()

        total_loss += loss.item()
        total_acc += acc
        n_batches += 1
        pbar.set_postfix(loss=f'{total_loss/n_batches:.4f}', acc=f'{total_acc/n_batches:.4f}')

    if n_batches == 0:
        return float('nan'), float('nan')
    return total_loss / n_batches, total_acc / n_batches

# Use full configured epochs (cap removed)
CFG['epochs'] = CFG.get('epochs', 10)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')
best_state = copy.deepcopy(model.state_dict())

print(f"Training for {CFG['epochs']} epochs on {DEVICE}...")
for epoch in range(1, CFG['epochs'] + 1):
    t0 = time.time()

    tr_loss, tr_acc = run_epoch(
        model, train_dl, optimizer=optimizer,
        grad_clip=CFG.get('grad_clip', 1.0),
        desc=f'Epoch {epoch} Train'
    )

    vl_loss, vl_acc = run_epoch(
        model, val_dl, optimizer=None,
        desc=f'Epoch {epoch} Val'
    )

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    elapsed = time.time() - t0
    print(
        f"Epoch {epoch:2d}/{CFG['epochs']} | "
        f"train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | "
        f"val_loss={vl_loss:.4f} val_acc={vl_acc:.4f} | {elapsed:.1f}s"
    )

    if np.isfinite(vl_loss) and vl_loss < best_val_loss:
        best_val_loss = vl_loss
        best_state = copy.deepcopy(model.state_dict())
        print(f"  ✓ New best model saved (val_loss={best_val_loss:.4f})")

model.load_state_dict(best_state)
print("\nTraining complete. Best val_loss:", best_val_loss)

In [ ]:

# ── Plot training curves ──────────────────────────────────────────────────────
if 'history' not in globals():
    raise RuntimeError("history not found. Run the training loop cell first.")

required_keys = ['train_loss', 'val_loss', 'train_acc', 'val_acc']
for k in required_keys:
    if k not in history:
        raise RuntimeError(f"history['{k}'] is missing.")

n_done = min(len(history['train_loss']),
             len(history['val_loss']),
             len(history['train_acc']),
             len(history['val_acc']))

if n_done == 0:
    raise RuntimeError("No epoch results in history. Training may not have run.")

epochs_range = range(1, n_done + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
axes[0].plot(epochs_range, history['train_loss'][:n_done], label='Train', marker='o')
axes[0].plot(epochs_range, history['val_loss'][:n_done],   label='Val',   marker='s')
axes[0].set_title('Cross-Entropy Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, history['train_acc'][:n_done], label='Train', marker='o')
axes[1].plot(epochs_range, history['val_acc'][:n_done],   label='Val',   marker='s')
axes[1].set_title('Token Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Plotted {n_done} epochs. Saved: training_curves.png")

## 7. BFGS Constant Optimisation

After the GPT decoder generates a **skeleton equation** (with `<C>` placeholders),
we optimise the constant values using the **Broyden–Fletcher–Goldfarb–Shanno (BFGS)**  
algorithm, minimising the MSE on the input point cloud.

This decouples structural learning (model) from numerical fitting (BFGS).

In [ ]:
def skeleton_to_callable(skeleton: str, n_vars: int):
    """
    Replace <C> placeholders with sympy symbols C0, C1, … and
    return a callable f(X, consts) → numpy array.
    """
    var_syms   = [sp.Symbol(f'x{i+1}') for i in range(n_vars)]
    c_count    = skeleton.count('<C>')
    const_syms = [sp.Symbol(f'C{i}') for i in range(c_count)]

    # Replace each <C> with its symbol
    filled = skeleton
    for i, cs in enumerate(const_syms):
        filled = filled.replace('<C>', str(cs), 1)

    try:
        expr = sp.sympify(filled, locals={str(s): s for s in var_syms + const_syms})
    except Exception:
        return None, 0

    all_syms = var_syms + const_syms
    func     = sp.lambdify(all_syms, expr, modules='numpy')

    def evaluate(X: np.ndarray, consts: np.ndarray) -> np.ndarray:
        # X: [N, n_vars],  consts: [c_count]
        args = [X[:, i] for i in range(n_vars)] + list(consts)
        try:
            result = func(*args)
            if np.isscalar(result):
                result = np.full(len(X), result)
            return np.where(np.isfinite(result), result, 1e10)
        except Exception:
            return np.full(len(X), 1e10)

    return evaluate, c_count


def bfgs_fit_constants(skeleton: str,
                       X: np.ndarray,
                       y: np.ndarray,
                       n_vars: int,
                       n_restarts: int = 5) -> tuple:
    """
    Optimise the constants in a skeleton equation using BFGS.

    Returns
    -------
    best_consts : np.ndarray  (optimised constant values)
    best_mse   : float
    """
    evaluate, c_count = skeleton_to_callable(skeleton, n_vars)
    if evaluate is None or c_count == 0:
        return np.array([]), np.inf

    def mse_objective(consts):
        y_pred = evaluate(X, consts)
        resid  = y_pred - y
        return np.mean(resid ** 2)

    best_consts = np.zeros(c_count)
    best_mse    = np.inf

    for _ in range(n_restarts):
        x0 = np.random.uniform(-3.0, 3.0, c_count)
        try:
            res = minimize(mse_objective, x0, method='BFGS',
                           options={'maxiter': 1000, 'gtol': 1e-8})
            if res.fun < best_mse:
                best_mse    = res.fun
                best_consts = res.x
        except Exception:
            continue

    return best_consts, best_mse


def fill_skeleton(skeleton: str, consts: np.ndarray) -> str:
    """Replace <C> tokens with their optimised float values."""
    result = skeleton
    for c in consts:
        result = result.replace('<C>', f'{c:.4f}', 1)
    return result


# ── Test BFGS ─────────────────────────────────────────────────────────────────
skeleton_test = 'sin(<C> * x1) + <C>'
X_test = np.random.uniform(-3, 3, (50, 1))
y_test = np.sin(1.5 * X_test[:, 0]) + 0.8
consts, mse = bfgs_fit_constants(skeleton_test, X_test, y_test, n_vars=1)
print(f'Skeleton : {skeleton_test}')
print(f'Fitted   : {fill_skeleton(skeleton_test, consts)}')
print(f'True     : sin(1.5*x1) + 0.8')
print(f'MSE      : {mse:.6f}')

## 8. Evaluation Pipeline

We report four metrics following the NeurIPS ML4PS 2025 paper:

| Metric | Description |
|---|---|
| **R²** | Coefficient of determination on hold-out points |
| **RMSE** | Root mean-squared error |
| **Token Accuracy** | % of equation tokens predicted correctly |
| **Exact Match %** | % of equations that are symbolically identical to ground truth |

In [ ]:
def r2_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2) + 1e-10
    return float(1.0 - ss_res / ss_tot)

def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def normalised_mse(y_true: np.ndarray, y_pred: np.ndarray, eps: float = 1e-6) -> float:
    norm = np.linalg.norm(y_true + eps) ** 2 + 1e-10
    return float(np.sum((y_true - y_pred) ** 2) / norm)

def filter_special_tokens(tokens: list[str]) -> list[str]:
    return [t for t in tokens if t not in ('<SOS>', '<EOS>', '<PAD>')]

def symbolic_match(pred_str: str, true_str: str) -> bool:
    try:
        syms = {f'x{i+1}': sp.Symbol(f'x{i+1}') for i in range(5)}
        expr1 = sp.sympify(pred_str, locals=syms)
        expr2 = sp.sympify(true_str, locals=syms)
        return sp.simplify(expr1 - expr2) == 0
    except Exception:
        return False

@torch.no_grad()
def evaluate_model(model, dataset, n_samples=200, n_vars=2, bfgs=True, noise_std=0.0):
    model.eval()

    # SAFE top_k: must be <= vocab size
    safe_top_k = max(1, min(40, VOCAB_SIZE - 1))
    sampler = PointCloudSampler(
        n_points=100,
        x_range=(-3.0, 3.0),
        n_vars=n_vars,
        noise_std=noise_std,
    )

    results = []
    if len(dataset) == 0:
        return {
            'n_total': 0, 'n_valid': 0,
            'r2_mean': np.nan, 'rmse_mean': np.nan, 'nmse_mean': np.nan,
            'token_acc_mean': np.nan, 'exact_match': np.nan, 'details': []
        }

    indices = random.sample(range(len(dataset)), min(n_samples, len(dataset)))

    for idx in tqdm(indices, desc='Evaluating'):
        item = dataset.data[idx]
        eq_str = item['eq_str']
        prefix = item['prefix']

        X_test, y_test = sampler.sample(eq_str)
        if X_test is None or y_test is None:
            continue

        cloud_t = torch.tensor(
            np.concatenate([X_test, y_test.reshape(-1, 1)], axis=1)[np.newaxis],
            dtype=torch.float32,
            device=DEVICE
        )

        # generate with safe top_k
        gen_ids = model.generate(cloud_t, max_new_tokens=80, top_k=safe_top_k)
        gen_tokens = ids_to_tokens(gen_ids)
        gen_pfx_toks = filter_special_tokens(gen_tokens)
        gen_prefix = ' '.join(gen_pfx_toks)

        gen_infix = prefix_tokens_to_infix(gen_pfx_toks)

        true_tokens = prefix.split()
        pred_tokens = gen_pfx_toks
        n_match = sum(p == g for p, g in zip(pred_tokens, true_tokens))
        n_total = max(len(true_tokens), 1)
        tok_acc = n_match / n_total

        if bfgs and '<C>' in gen_infix:
            consts, _ = bfgs_fit_constants(gen_infix, X_test, y_test, n_vars=n_vars)
            pred_eq = fill_skeleton(gen_infix, consts)
        else:
            pred_eq = gen_infix  # use expression directly (fixed: was replacing <C> with 1.0)

        try:
            evaluate_fn, _ = skeleton_to_callable(pred_eq, n_vars)
            if evaluate_fn is not None:
                y_pred = evaluate_fn(X_test, np.array([]))
                r2 = r2_score(y_test, y_pred)
                rms = rmse(y_test, y_pred)
                nmse = normalised_mse(y_test, y_pred)
            else:
                r2 = rms = nmse = float('nan')
        except Exception:
            r2 = rms = nmse = float('nan')

        exact = symbolic_match(pred_eq, eq_str)

        results.append(dict(
            eq_str=eq_str,
            prefix=prefix,
            pred_prefix=gen_prefix,
            pred_eq=pred_eq,
            r2=r2,
            rmse=rms,
            nmse=nmse,
            tok_acc=tok_acc,
            exact=exact,
        ))

    finite = [r for r in results if np.isfinite(r['r2'])]

    agg = {
        'n_total': len(results),
        'n_valid': len(finite),
        'r2_mean': np.nanmean([r['r2'] for r in finite]) if finite else np.nan,
        'rmse_mean': np.nanmean([r['rmse'] for r in finite]) if finite else np.nan,
        'nmse_mean': np.nanmean([r['nmse'] for r in finite]) if finite else np.nan,
        'token_acc_mean': np.nanmean([r['tok_acc'] for r in results]) if results else np.nan,
        'exact_match': np.mean([r['exact'] for r in results]) if results else np.nan,
        'details': results,
    }
    return agg


print('Evaluating on validation set...')
eval_results = evaluate_model(model, val_ds, n_samples=200, n_vars=CFG['n_vars'])

print('\n' + '='*55)
print('         EVALUATION RESULTS (validation set)')
print('='*55)
print(f"  Samples evaluated : {eval_results['n_total']}")
print(f"  Valid predictions : {eval_results['n_valid']}")
print(f"  R²  (mean)        : {eval_results['r2_mean']:.4f}")
print(f"  RMSE (mean)       : {eval_results['rmse_mean']:.4f}")
print(f"  NMSE (mean)       : {eval_results['nmse_mean']:.4f}")
print(f"  Token Accuracy    : {eval_results['token_acc_mean']*100:.2f}%")
print(f"  Exact Match       : {eval_results['exact_match']*100:.2f}%")
print('='*55)

In [ ]:
# ── Curriculum training run (stable + filtered + safe top_k) ──────────────────
# Prereqs expected in notebook state:
# - CATALOGUE, SRDataset, DataLoader, collate_fn
# - SymbolicGPT, run_epoch, evaluate_model
# - VOCAB_SIZE, PAD_IDX, CFG, DEVICE

def _canon_formula(eq: str, n_vars: int) -> str:
    try:
        expr = sp.sympify(eq)
        syms = sorted(list(expr.free_symbols), key=lambda s: str(s))[:n_vars]
        repl = {s: sp.Symbol(f'x{i+1}') for i, s in enumerate(syms)}
        return str(expr.xreplace(repl))
    except Exception:
        return eq

def _is_tokenizable(entry: dict, n_vars: int, max_len: int = 80) -> bool:
    try:
        eq = _canon_formula(entry['formula'], n_vars)
        pfx = equation_str_to_prefix(eq, n_vars=n_vars)
        return ('<UNK>' not in pfx) and (len(pfx) <= max_len)
    except Exception:
        return False

def _prepare_stage_catalogue(catalogue, n_vars: int):
    prepped = []
    for e in catalogue:
        if not e.get('data_path'):
            continue
        ev = e.get('n_vars')
        if n_vars == 4:
            if ev not in (4, 5):
                continue
        else:
            if ev not in (None, n_vars):
                continue
        if not _is_tokenizable(e, n_vars):
            continue
        e2 = dict(e)
        e2['formula'] = _canon_formula(e2['formula'], n_vars)
        prepped.append(e2)
    return prepped

# Use full configured epochs (cap removed) (quick + non-hanging)
curriculum_stages = [
    {'n_vars': 2, 'epochs': 2, 'lr': 3e-4, 'train_size': 600, 'batch_size': 32},
    {'n_vars': 3, 'epochs': 2, 'lr': 1e-4, 'train_size': 500, 'batch_size': 32},
    {'n_vars': 4, 'epochs': 2, 'lr': 5e-5, 'train_size': 400, 'batch_size': 24},
]

print("\n" + "="*80)
print("STARTING CURRICULUM TRAINING (STABLE MODE)")
print("="*80)

# ensure evaluate uses valid top_k with your vocab size
SAFE_TOP_K = max(1, min(20, VOCAB_SIZE - 1))
print(f"[Curriculum] Using safe top_k={SAFE_TOP_K} for generation-based evaluation")

curriculum_history = {'stages': [], 'evaluations': []}

for stage_idx, stage in enumerate(curriculum_stages, start=1):
    n_vars = stage['n_vars']
    epochs = stage['epochs']
    lr = stage['lr']
    train_size = stage['train_size']
    batch_size = stage['batch_size']

    stage_catalogue = _prepare_stage_catalogue(CATALOGUE, n_vars=n_vars)
    print(f"\nStage {stage_idx}: n_vars={n_vars} | usable equations={len(stage_catalogue)}")

    if len(stage_catalogue) == 0:
        print("  ⚠️ No usable equations, skipping.")
        curriculum_history['stages'].append({
            'n_vars': n_vars,
            'history': {'train_loss': [], 'train_acc': []}
        })
        curriculum_history['evaluations'].append({
            'after_stage': stage_idx, 'n_vars': n_vars, 'results': {}
        })
        continue

    stage_ds = SRDataset(
        size=min(train_size, max(120, len(stage_catalogue) * 25)),
        n_vars=n_vars,
        n_points=CFG.get('n_points', 50),
        max_depth=CFG.get('max_depth', 3),
        feynman_catalogue=stage_catalogue,
        enable_augmentation=True,
        aug_prob_var_permute=0.3,
        aug_prob_noise=0.3,
        aug_prob_subsample=0.2,
        aug_prob_const_scale=0.2,
        aug_prob_const_shift=0.2,
    )
    stage_dl = DataLoader(
        stage_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, num_workers=0
    )

    sample_clouds, _, _, _, _ = next(iter(stage_dl))
    in_dim_runtime = int(sample_clouds.shape[-1])

    # reinit stage model to match n_vars+1 cloud shape
    model = SymbolicGPT(
        in_dim=in_dim_runtime,
        vocab_size=VOCAB_SIZE,
        emb_dim=CFG['emb_dim'],
        n_heads=CFG['n_heads'],
        n_layers=CFG['n_layers'],
        max_seq_len=CFG['max_seq_len'],
        dropout=CFG['dropout'],
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=CFG.get('weight_decay', 1e-4)
    )
    total_steps = max(epochs * len(stage_dl), 1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=total_steps, eta_min=lr / 10
    )
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

    # bind globals used by run_epoch in your notebook
    globals()['scheduler'] = scheduler
    globals()['criterion'] = criterion

    stage_history = {'train_loss': [], 'train_acc': []}
    for ep in range(1, epochs + 1):
        tr_loss, tr_acc = run_epoch(
            model, stage_dl, optimizer=optimizer,
            grad_clip=CFG.get('grad_clip', 1.0),
            desc=f"Stage {n_vars}-var Ep {ep}/{epochs}"
        )
        stage_history['train_loss'].append(float(tr_loss))
        stage_history['train_acc'].append(float(tr_acc))
        print(f"  Epoch {ep}/{epochs}: loss={tr_loss:.4f}, acc={tr_acc:.4f}")

    curriculum_history['stages'].append({'n_vars': n_vars, 'history': stage_history})

    # lightweight generalization checks
    eval_results = {}
    for nv in [1, 2, 3, 4]:
        eval_cat = _prepare_stage_catalogue(CATALOGUE, n_vars=nv)
        if len(eval_cat) == 0:
            continue

        val_ds = SRDataset(
            size=min(100, max(30, len(eval_cat) * 6)),
            n_vars=nv,
            n_points=30,
            max_depth=CFG.get('max_depth', 3),
            feynman_catalogue=eval_cat,
            enable_augmentation=False,
        )

        # wrap evaluate_model call safely; if your evaluate_model doesn't expose top_k,
        # it should still work if you've already patched safe top_k internally.
        try:
            res = evaluate_model(model, val_ds, n_samples=min(50, len(val_ds)), n_vars=nv)
            eval_results[nv] = res
            print(f"  Eval {nv}-var: R²={res['r2_mean']:.4f}, TokAcc={res['token_acc_mean']*100:.2f}%")
        except Exception as exc:
            print(f"  Eval {nv}-var skipped: {exc}")

    curriculum_history['evaluations'].append({
        'after_stage': stage_idx,
        'n_vars': n_vars,
        'results': eval_results
    })

torch.save(model.state_dict(), 'symbolicgpt_curriculum_final.pt')
print('\n✅ Model saved to symbolicgpt_curriculum_final.pt')

# ── Plot curriculum history ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
for stage_data in curriculum_history['stages']:
    n_vars = stage_data['n_vars']
    losses = stage_data['history']['train_loss']
    if losses:
        ax.plot(losses, marker='o', label=f'{n_vars}-var stage', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss')
ax.set_title('Training Loss Per Curriculum Stage')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
for stage_data in curriculum_history['stages']:
    n_vars = stage_data['n_vars']
    accs = stage_data['history']['train_acc']
    if accs:
        ax.plot(accs, marker='s', label=f'{n_vars}-var stage', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Token Accuracy')
ax.set_title('Token Accuracy Per Curriculum Stage')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
for eval_data in curriculum_history['evaluations']:
    stage_num = eval_data['after_stage']
    results = eval_data['results']
    if not results:
        continue
    n_vars_list = sorted(results.keys())
    r2_vals = [results[nv]['r2_mean'] for nv in n_vars_list]
    ax.plot(n_vars_list, r2_vals, marker='o', label=f'After Stage {stage_num}', linewidth=2)
ax.set_xlabel('Number of Variables')
ax.set_ylabel('R² Score')
ax.set_title('Generalization Across Variable Counts')
ax.set_xticks([1, 2, 3, 4])
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
for eval_data in curriculum_history['evaluations']:
    stage_num = eval_data['after_stage']
    results = eval_data['results']
    if not results:
        continue
    n_vars_list = sorted(results.keys())
    exact_vals = [results[nv]['exact_match'] * 100 for nv in n_vars_list]
    ax.plot(n_vars_list, exact_vals, marker='s', label=f'After Stage {stage_num}', linewidth=2)
ax.set_xlabel('Number of Variables')
ax.set_ylabel('Exact Match (%)')
ax.set_title('Exact Match Rate Across Variable Counts')
ax.set_xticks([1, 2, 3, 4])
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('curriculum_learning_results.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n✅ Curriculum learning complete! Results saved.')

## 9. Qualitative Results – Prediction Examples

### Decoding Strategy

* Greedy decoding:
  * Stable and deterministic
  * May miss better equations

* Top-k sampling:
  * More diverse outputs
  * Helps explore multiple candidate equations

We use both to balance stability and exploration.

### Why This Approach is Better

* Produces interpretable equations (unlike black-box models)
* Trained once → reusable across datasets
* Faster inference compared to genetic programming
* Combines neural learning + symbolic optimization

### Failure Cases

* Structurally incorrect equations but numerically close
* Missing terms or incorrect operators
* Sensitivity to noisy data

Insight:
Neural models capture local structure well but struggle with global symbolic correctness.

### Baseline Comparison

As a simple baseline, we also fit a linear regression model on the same `(X, y)` pairs.
This helps contrast symbolic model behavior with a purely linear predictor.


In [ ]:
# ── Hybrid Transformer + Genetic Programming ──────────────────────────────────
from typing import List, Tuple, Optional, Dict


class HybridTransformerGP:
    """Combine SymbolicGPT generation with Genetic Programming refinement."""

    def __init__(self,
                 transformer_model,
                 population_size: int = 50,
                 generations: int = 20,
                 tournament_size: int = 3,
                 mutation_rate: float = 0.15,
                 crossover_rate: float = 0.8,
                 n_restarts: int = 5,
                 plateau_patience: int = 5,
                 max_tokens: int = 100):
        self.model = transformer_model
        self.population_size = population_size
        self.generations = generations
        self.tournament_size = tournament_size
        self.mutation_rate = mutation_rate
        self.crossover_rate = crossover_rate
        self.n_restarts = n_restarts
        self.plateau_patience = plateau_patience
        self.max_tokens = max_tokens
        self._fitness_cache: Dict[Tuple[str, int], float] = {}

    def _tokens_to_equation(self, token_ids) -> Optional[str]:
        try:
            toks = ids_to_tokens(token_ids)
            toks = [t for t in toks if t not in ('<SOS>', '<EOS>', '<PAD>')]
            if len(toks) == 0 or len(toks) > self.max_tokens:
                return None
            expr = tokens_to_sympy(toks)
            if expr is None:
                return None
            eq = str(expr)
            if len(eq) == 0 or len(eq) > 2000:
                return None
            return eq
        except Exception:
            return None

    def generate_initial_population(self, X, y) -> List[str]:
        candidates = set()
        cloud = np.concatenate([X, y.reshape(-1, 1)], axis=1).astype(np.float32)
        cloud_t = torch.tensor(cloud[np.newaxis], dtype=torch.float32, device=DEVICE)

        temperatures = np.linspace(0.3, 1.5, num=max(6, self.population_size // 6))

        for temp in temperatures:
            for _ in range(max(2, self.population_size // len(temperatures))):
                try:
                    gen_ids = self.model.generate(
                        cloud_t,
                        max_new_tokens=min(80, self.max_tokens),
                        temperature=float(temp),
                        top_k=40,
                    )
                    eq = self._tokens_to_equation(gen_ids)
                    if eq is not None:
                        candidates.add(eq)
                except Exception:
                    continue

        if len(candidates) == 0:
            candidates.add('x1')

        candidates = list(candidates)
        random.shuffle(candidates)

        if len(candidates) >= self.population_size:
            return candidates[:self.population_size]

        while len(candidates) < self.population_size:
            candidates.append(random.choice(candidates))

        return candidates

    def _safe_evaluate(self, eq_str, X, y, n_vars) -> Optional[float]:
        try:
            eval_fn, _ = skeleton_to_callable(eq_str, n_vars)
            if eval_fn is None:
                return None
            y_pred = eval_fn(X, EMPTY_PARAMS)
            if not np.all(np.isfinite(y_pred)):
                return None
            return r2_score(y, y_pred)
        except Exception:
            return None

    def fitness(self, eq_str, X, y, n_vars) -> float:
        cache_key = (eq_str, n_vars)
        if cache_key in self._fitness_cache:
            return self._fitness_cache[cache_key]

        try:
            eq_opt = optimize_constants(eq_str, X, y, n_vars=n_vars, n_restarts=self.n_restarts)
            r2 = self._safe_evaluate(eq_opt, X, y, n_vars)
            if r2 is None or not np.isfinite(r2):
                self._fitness_cache[cache_key] = 0.0
                return 0.0

            complexity_penalty = 0.01 * len(eq_opt)
            fit = float(r2 - complexity_penalty)
            fit = max(fit, 0.0)
            self._fitness_cache[cache_key] = fit
            return fit
        except Exception:
            self._fitness_cache[cache_key] = 0.0
            return 0.0

    def crossover(self, parent1_str, parent2_str) -> str:
        try:
            expr1 = sp.sympify(parent1_str)
            expr2 = sp.sympify(parent2_str)
            nodes1 = [n for n in sp.preorder_traversal(expr1)]
            nodes2 = [n for n in sp.preorder_traversal(expr2)]
            if not nodes1 or not nodes2:
                return parent1_str

            sub1 = random.choice(nodes1)
            sub2 = random.choice(nodes2)
            child = expr1.xreplace({sub1: sub2})
            child = sp.simplify(child)
            child_str = str(child)
            if len(child_str) > 2000:
                return parent1_str
            return child_str
        except Exception:
            return parent1_str

    def mutate(self, eq_str, mutation_rate=None, n_vars=2) -> str:
        rate = self.mutation_rate if mutation_rate is None else mutation_rate
        if random.random() > rate:
            return eq_str

        try:
            expr = sp.sympify(eq_str)
            nodes = [n for n in sp.preorder_traversal(expr)]
            if not nodes:
                return eq_str

            node = random.choice(nodes)
            var_syms = [sp.Symbol(f'x{i+1}') for i in range(n_vars)]

            replacement = node
            if isinstance(node, sp.Symbol):
                choices = [v for v in var_syms if v != node]
                if choices:
                    replacement = random.choice(choices)
            elif isinstance(node, sp.Number):
                replacement = sp.Float(float(node) + np.random.normal(0.0, 0.5))
            elif getattr(node, 'func', None) in (sp.sin, sp.cos):
                replacement = (sp.cos(node.args[0]) if node.func == sp.sin else sp.sin(node.args[0]))
            elif isinstance(node, sp.Add) and len(node.args) == 2:
                replacement = node.args[0] - node.args[1]
            elif isinstance(node, sp.Mul) and len(node.args) == 2:
                replacement = node.args[0] / (node.args[1] + sp.Float(1e-6))

            mutated = expr.xreplace({node: replacement})
            mutated = sp.simplify(mutated)
            mutated_str = str(mutated)
            if len(mutated_str) > 2000:
                return eq_str
            return mutated_str
        except Exception:
            return eq_str

    def tournament_selection(self, population, fitness_scores, k, tournament_size) -> List[str]:
        selected = []
        n = len(population)
        if n == 0:
            return selected

        for _ in range(k):
            idxs = np.random.choice(n, size=min(tournament_size, n), replace=False)
            best_idx = max(idxs, key=lambda i: fitness_scores[i])
            selected.append(population[int(best_idx)])

        return selected

    def evolve(self, X, y, n_vars, verbose=True) -> Tuple[str, float, List[Dict]]:
        self._fitness_cache = {}
        history: List[Dict] = []

        population = self.generate_initial_population(X, y)
        best_eq = population[0] if population else 'x1'
        best_fit = -np.inf

        plateau_count = 0

        for gen in range(self.generations):
            fitness_scores = [self.fitness(eq, X, y, n_vars) for eq in population]

            gen_best_idx = int(np.argmax(fitness_scores))
            gen_best_fit = float(fitness_scores[gen_best_idx])
            gen_best_eq = population[gen_best_idx]
            gen_mean_fit = float(np.mean(fitness_scores)) if fitness_scores else 0.0

            if gen_best_fit > best_fit + 1e-10:
                best_fit = gen_best_fit
                best_eq = gen_best_eq
                plateau_count = 0
            else:
                plateau_count += 1

            history.append({
                'gen': gen,
                'best_fitness': gen_best_fit,
                'mean_fitness': gen_mean_fit,
                'best_eq': gen_best_eq,
            })

            if verbose:
                print(
                    f"Gen {gen:02d} | best={gen_best_fit:.4f} | "
                    f"mean={gen_mean_fit:.4f} | eq={gen_best_eq[:80]}"
                )

            if plateau_count >= self.plateau_patience:
                if verbose:
                    print(f'Early stopping: fitness plateau for {self.plateau_patience} generations.')
                break

            n_elite = max(1, int(0.10 * self.population_size))
            elite_indices = np.argsort(fitness_scores)[-n_elite:]
            elites = [population[i] for i in elite_indices]

            selected = self.tournament_selection(
                population,
                fitness_scores,
                k=max(self.population_size - n_elite, 1),
                tournament_size=self.tournament_size,
            )

            offspring = []
            while len(offspring) < self.population_size - n_elite:
                p1 = random.choice(selected) if selected else random.choice(population)
                p2 = random.choice(selected) if selected else random.choice(population)

                child = p1
                if random.random() < self.crossover_rate:
                    child = self.crossover(p1, p2)

                child = self.mutate(child, mutation_rate=self.mutation_rate, n_vars=n_vars)
                offspring.append(child)

            population = elites + offspring

        return best_eq, float(best_fit if np.isfinite(best_fit) else 0.0), history

In [ ]:
# ── Hybrid GP test (robust, self-contained) ───────────────────────────────────
import numpy as np
import sympy as sp
import torch
import matplotlib.pyplot as plt

# 1) missing helper (defined here so cell is self-contained)
def tokens_to_sympy(tokens: list[str]):
    try:
        if tokens is None:
            return None
        toks = [t for t in tokens if t not in ('<SOS>', '<EOS>', '<PAD>', '<UNK>')]
        if len(toks) == 0:
            return None
        infix = prefix_tokens_to_infix(toks)
        if not infix:
            return None
        syms = {f'x{i+1}': sp.Symbol(f'x{i+1}') for i in range(5)}
        return sp.sympify(infix, locals=syms)
    except Exception:
        return None

print("\n" + "="*80)
print("TESTING HYBRID TRANSFORMER + GP INTEGRATION")
print("="*80 + "\n")

# 2) ensure model dimension matches this test (2 vars => in_dim=3)
TEST_N_VARS = 2
TEST_IN_DIM = TEST_N_VARS + 1
SAFE_TOP_K = max(1, min(20, VOCAB_SIZE - 1))

model_gp = SymbolicGPT(
    in_dim=TEST_IN_DIM,
    vocab_size=VOCAB_SIZE,
    emb_dim=CFG['emb_dim'],
    n_heads=CFG['n_heads'],
    n_layers=CFG['n_layers'],
    max_seq_len=CFG['max_seq_len'],
    dropout=CFG['dropout'],
).to(DEVICE)

# optional partial weight transfer from current model
if 'model' in globals():
    src = model.state_dict()
    dst = model_gp.state_dict()
    compatible = {k: v for k, v in src.items() if k in dst and dst[k].shape == v.shape}
    dst.update(compatible)
    model_gp.load_state_dict(dst, strict=False)
    print(f"Loaded {len(compatible)} compatible tensors into test model.")

# 3) synthetic benchmark
X_gp_test = np.random.uniform(-3, 3, (200, TEST_N_VARS)).astype(np.float32)
y_gp_test = (np.sin(2 * X_gp_test[:, 0]) + X_gp_test[:, 1]**2).astype(np.float32)

print("Ground truth equation: sin(2*x1) + x2^2")
print("Testing hybrid approach...\n")

hybrid_gp = HybridTransformerGP(
    transformer_model=model_gp,
    population_size=30,
    generations=15,
    mutation_rate=0.15,
)

best_eq_gp, best_fit_gp, history_gp = hybrid_gp.evolve(
    X_gp_test, y_gp_test, n_vars=TEST_N_VARS, verbose=True
)

print(f"\n{'='*80}")
print("RESULTS:")
print(f"{'='*80}")
print(f"Best equation found: {best_eq_gp}")
print(f"Final fitness: {best_fit_gp:.4f}")

# 4) pure transformer baseline
cloud_gp_test = np.concatenate([X_gp_test, y_gp_test.reshape(-1, 1)], axis=1)
cloud_gp_test_t = torch.tensor(cloud_gp_test[np.newaxis], dtype=torch.float32, device=DEVICE)

transformer_tokens = model_gp.generate(cloud_gp_test_t, temperature=0.7, top_k=SAFE_TOP_K)
transformer_expr = tokens_to_sympy(ids_to_tokens(transformer_tokens))
transformer_eq_str = str(transformer_expr) if transformer_expr is not None else "INVALID"

print(f"\nPure transformer result: {transformer_eq_str}")

# 5) evaluate both
EMPTY_PARAMS_LOCAL = np.array([], dtype=np.float32)

eval_fn_gp = None
y_pred_gp = None
r2_gp = None
if best_eq_gp:
    eval_fn_gp, _ = skeleton_to_callable(best_eq_gp, TEST_N_VARS)
    if eval_fn_gp is not None:
        y_pred_gp = eval_fn_gp(X_gp_test, EMPTY_PARAMS_LOCAL)
        r2_gp = r2_score(y_gp_test, y_pred_gp)
        print(f"Hybrid GP R²: {r2_gp:.4f}")

eval_fn_trans = None
y_pred_trans = None
r2_trans = None
if transformer_eq_str != "INVALID":
    eval_fn_trans, _ = skeleton_to_callable(transformer_eq_str, TEST_N_VARS)
    if eval_fn_trans is not None:
        y_pred_trans = eval_fn_trans(X_gp_test, EMPTY_PARAMS_LOCAL)
        r2_trans = r2_score(y_gp_test, y_pred_trans)
        print(f"Pure Transformer R²: {r2_trans:.4f}")

# 6) plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

gens = [h['gen'] for h in history_gp]
best_fits = [h['best_fitness'] for h in history_gp]
mean_fits = [h['mean_fitness'] for h in history_gp]

ax1.plot(gens, best_fits, 'b-o', label='Best Fitness', linewidth=2, markersize=4)
ax1.plot(gens, mean_fits, 'r--s', label='Mean Fitness', linewidth=1.5, markersize=3)
ax1.set_xlabel('Generation', fontsize=12)
ax1.set_ylabel('Fitness', fontsize=12)
ax1.set_title('GP Evolution Progress', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

idx = np.arange(min(100, len(y_gp_test)))
ax2.plot(idx, y_gp_test[:len(idx)], 'k-', label='Ground Truth', linewidth=2)
if y_pred_gp is not None and r2_gp is not None:
    ax2.plot(idx, y_pred_gp[:len(idx)], 'b--', label=f'Hybrid GP (R²={r2_gp:.3f})', linewidth=1.5)
if y_pred_trans is not None and r2_trans is not None:
    ax2.plot(idx, y_pred_trans[:len(idx)], 'r:', label=f'Transformer (R²={r2_trans:.3f})', linewidth=1.5)

ax2.set_xlabel('Sample Index', fontsize=12)
ax2.set_ylabel('y', fontsize=12)
ax2.set_title('Prediction Comparison', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('hybrid_gp_results.png', dpi=120, bbox_inches='tight')
plt.show()

print("\n✅ Hybrid GP integration test complete!")

In [ ]:
print("\nSample predictions (first 5):")
print("-" * 80)

if 'eval_results' not in globals():
    raise RuntimeError("eval_results not found. Run evaluation cell first.")

# robust extraction
details = []
if isinstance(eval_results, dict):
    if 'details' in eval_results:
        details = eval_results['details']
    elif 'results' in eval_results and isinstance(eval_results['results'], list):
        details = eval_results['results']

if not details:
    print("No per-sample details found in eval_results.")
    print("Available keys:", list(eval_results.keys()) if isinstance(eval_results, dict) else type(eval_results))
else:
    for r in details[:5]:
        eq_true = r.get('eq_str', 'NA')
        pfx_true = r.get('prefix', 'NA')
        pfx_pred = r.get('pred_prefix', 'NA')
        eq_pred = r.get('pred_eq', 'NA')
        r2 = r.get('r2', float('nan'))
        rmse_v = r.get('rmse', float('nan'))
        tok = r.get('tok_acc', float('nan'))
        exact = r.get('exact', False)

        print(f"True equation    : {eq_true}")
        print(f"True prefix      : {pfx_true}")
        print(f"Pred prefix      : {pfx_pred}")
        print(f"Pred equation    : {eq_pred}")
        print(f"R²={r2:.4f}  RMSE={rmse_v:.4f}  TokAcc={tok*100:.1f}%  Exact={exact}")
        print("-" * 80)

In [ ]:
# ── Visualise 1-D predictions ─────────────────────────────────────────────────
# Filter to 1-variable predictions for easy plotting
# (retrain or use the saved model if n_vars=1 was trained)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

plot_count = 0
for r in details:
    if plot_count >= 6:
        break
    if not np.isfinite(r.get('r2', float('nan'))):
        continue

    eq_str   = r['eq_str']
    pred_eq  = r['pred_eq']
    n_v      = CFG['n_vars']

    x_plot = np.linspace(-4, 4, 200)
    X_plot = np.column_stack([x_plot] + [np.zeros_like(x_plot)] * (n_v - 1))

    try:
        var_syms = {f'x{i+1}': sp.Symbol(f'x{i+1}') for i in range(n_v)}
        true_fn  = sp.lambdify(list(var_syms.values()),
                               sp.sympify(eq_str, locals=var_syms), 'numpy')
        y_true   = true_fn(*[X_plot[:, i] for i in range(n_v)])

        pred_fn  = sp.lambdify(list(var_syms.values()),
                               sp.sympify(pred_eq, locals=var_syms), 'numpy')
        y_pred   = pred_fn(*[X_plot[:, i] for i in range(n_v)])

        if not (np.all(np.isfinite(y_true)) and np.all(np.isfinite(y_pred))):
            continue

        ax = axes[plot_count]
        ax.plot(x_plot, y_true, 'b-',  linewidth=2,   label='True')
        ax.plot(x_plot, y_pred, 'r--', linewidth=1.5, label='Predicted')
        ax.set_ylim(np.percentile(y_true, 1) - 1, np.percentile(y_true, 99) + 1)
        ax.set_title(f"R²={r['r2']:.3f}", fontsize=9)
        ax.set_xlabel('x1')
        ax.legend(fontsize=7)
        plot_count += 1
    except Exception:
        continue

for i in range(plot_count, 6):
    axes[i].axis('off')

plt.suptitle('True (blue) vs Predicted (red) equations', fontsize=12)
plt.tight_layout()
plt.savefig('predictions.png', dpi=120)
plt.show()

## 10. Noise-Robustness Experiment

Following the PIGP / SymbolicDPO paper (NeurIPS ML4PS 2024), we measure how  
performance degrades under Gaussian noise added to the target `y` values.

In [ ]:
noise_levels  = [0.0, 0.05, 0.1, 0.2, 0.3]
noise_results = {}

for σ in noise_levels:
    print(f'\nEvaluating noise_std={σ}…')

    # Build a small noisy dataset for evaluation
    noisy_ds = SRDataset(
        size=500,
        n_vars=CFG['n_vars'],
        n_points=CFG['n_points'],
        max_depth=CFG['max_depth'],
        noise_std=σ,
        feynman_catalogue=CATALOGUE,
    )

    res = evaluate_model(model, noisy_ds, n_samples=100,
                         n_vars=CFG['n_vars'], noise_std=σ)
    noise_results[σ] = res
    print(f'  R²={res["r2_mean"]:.4f}  '
          f'RMSE={res["rmse_mean"]:.4f}  '
          f'TokAcc={res["token_acc_mean"]*100:.2f}%  '
          f'Exact={res["exact_match"]*100:.2f}%')

# ── Summary table ─────────────────────────────────────────────────────────────
print('\n' + '='*65)
print(f'{"Noise σ":>10} | {"R²":>8} | {"RMSE":>8} | {"TokAcc%":>9} | {"Exact%":>7}')
print('-'*65)
for σ, res in noise_results.items():
    print(f'{σ:>10.2f} | '
          f'{res["r2_mean"]:>8.4f} | '
          f'{res["rmse_mean"]:>8.4f} | '
          f'{res["token_acc_mean"]*100:>9.2f} | '
          f'{res["exact_match"]*100:>7.2f}')
print('='*65)

In [ ]:
# ── Plot noise robustness ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

sigma_vals = list(noise_results.keys())
r2_vals    = [noise_results[s]['r2_mean']            for s in sigma_vals]
rmse_vals  = [noise_results[s]['rmse_mean']           for s in sigma_vals]
tok_vals   = [noise_results[s]['token_acc_mean']*100  for s in sigma_vals]

axes[0].plot(sigma_vals, r2_vals,   'bo-', linewidth=2)
axes[0].set_xlabel('Noise σ (fraction of std)')
axes[0].set_ylabel('R²')
axes[0].set_title('R² vs Noise Level')
axes[0].set_ylim(0, 1.05)

axes[1].plot(sigma_vals, rmse_vals, 'rs-', linewidth=2)
axes[1].set_xlabel('Noise σ (fraction of std)')
axes[1].set_ylabel('RMSE')
axes[1].set_title('RMSE vs Noise Level')

axes[2].plot(sigma_vals, tok_vals,  'g^-', linewidth=2)
axes[2].set_xlabel('Noise σ (fraction of std)')
axes[2].set_ylabel('Token Accuracy (%)')
axes[2].set_title('Token Accuracy vs Noise Level')

plt.tight_layout()
plt.savefig('noise_robustness.png', dpi=120)
plt.show()

## 11. AI Feynman Dataset Evaluation (Optional)

The **AI Feynman** dataset (Udrescu & Tegmark, 2020) contains 100 physics equations  
from undergraduate physics.  Below we load it from a public URL, parse a subset,  
and run our model on it.  If the download fails, the section is gracefully skipped.

In [ ]:
FEYNMAN_EQUATIONS = [
    # (name, expression in terms of x1, x2, ...)
    ('I.6.2a',   'exp(-x1**2/2)/sqrt(2*3.14159)'),
    ('I.12.5',   'x1**2 * x2'),
    ('I.18.14',  'x1 * x2 * x3 * sin(x4)'),
    ('I.39.1',   '1.5 * x1 * x2'),
    ('I.43.16',  'x1 * x2 * x3 / x4'),
    ('I.43.31',  'x1 * x2 * x3'),
    ('II.4.23',  'x1 / (4 * 3.14159 * x2 * x3)'),
    ('II.21.32', 'x1 / (4 * 3.14159 * x2 * x3 * (1 - x4))'),
    ('II.35.21', 'x1 * x2 * tanh(x2 * x3 / (x4 * x5))'),
    ('II.38.3',  'x1 * x2 * x3 / x4'),
]

print(f'AI Feynman subset: {len(FEYNMAN_EQUATIONS)} equations')

feynman_rows = []
for name, expr in FEYNMAN_EQUATIONS:
    # Determine actual variable count
    used_vars = len([v for v in ['x1', 'x2', 'x3', 'x4', 'x5'] if v in expr])
    sampler_i = PointCloudSampler(n_points=100, x_range=(0.5, 5.0),
                                  n_vars=used_vars, noise_std=0.0)
    X_f, y_f = sampler_i.sample(expr)
    if X_f is None:
        print(f'  SKIP {name} (failed to sample)')
        continue

    # Use our model (trained on 2-variable data) – zero-pad extra vars
    n_v_model = CFG['n_vars']
    X_padded  = np.zeros((len(X_f), n_v_model), dtype=np.float32)
    X_padded[:, :min(used_vars, n_v_model)] = X_f[:, :min(used_vars, n_v_model)]

    cloud_t = torch.tensor(
        np.concatenate([X_padded, y_f.reshape(-1, 1)], axis=1)[np.newaxis],
        dtype=torch.float32, device=DEVICE
    )

    # ── Generate prefix tokens → convert to infix ────────────────────────────
    gen_ids      = model.generate(cloud_t, max_new_tokens=80, top_k=40)
    gen_tokens   = ids_to_tokens(gen_ids)
    gen_pfx_toks = filter_special_tokens(gen_tokens)
    gen_infix    = prefix_tokens_to_infix(gen_pfx_toks)

    # ── BFGS constant fitting ────────────────────────────────────────────────
    if '<C>' in gen_infix:
        consts, mse = bfgs_fit_constants(
            gen_infix, X_padded, y_f, n_vars=n_v_model
        )
        pred_eq = fill_skeleton(gen_infix, consts)
    else:
        pred_eq = gen_infix  # use expression directly (fixed: was replacing <C> with 1.0)
        mse = np.inf

    # ── Evaluate ─────────────────────────────────────────────────────────────
    try:
        evaluate_fn, _ = skeleton_to_callable(pred_eq, n_v_model)
        if evaluate_fn is not None:
            y_pred = evaluate_fn(X_padded, np.array([]))
            r2_f   = r2_score(y_f, y_pred)
            rmse_f = rmse(y_f, y_pred)
        else:
            r2_f = rmse_f = float('nan')
    except Exception:
        r2_f = rmse_f = float('nan')

    feynman_rows.append({
        'name':     name,
        'true_eq':  expr,
        'pred_pfx': ' '.join(gen_pfx_toks),
        'pred_eq':  pred_eq,
        'r2':       r2_f,
        'rmse':     rmse_f,
    })

print('\nAI Feynman Results:')
print(f'{"Name":>12} | {"R²":>8} | {"RMSE":>10} | Predicted')
print('-' * 80)
for row in feynman_rows:
    print(f"{row['name']:>12} | {row['r2']:>8.4f} | "
          f"{row['rmse']:>10.4f} | {row['pred_eq'][:45]}")

r2_mean_f   = np.nanmean([r['r2']   for r in feynman_rows])
rmse_mean_f = np.nanmean([r['rmse'] for r in feynman_rows])
print('-' * 80)
print(f'  Mean R² = {r2_mean_f:.4f}   Mean RMSE = {rmse_mean_f:.4f}')


## 12. R² Distribution Visualisation

In [ ]:
r2_vals = [r['r2'] for r in eval_results['details'] if np.isfinite(r['r2'])]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of R²
axes[0].hist(r2_vals, bins=30, edgecolor='black', color='steelblue', alpha=0.8)
axes[0].axvline(np.mean(r2_vals), color='red', linestyle='--',
                linewidth=2, label=f'Mean = {np.mean(r2_vals):.3f}')
axes[0].set_xlabel('R²')
axes[0].set_ylabel('Count')
axes[0].set_title('R² Distribution (Validation Set)')
axes[0].legend()

# Cumulative distribution (as in SymbolicGPT Figure 2)
nmse_vals = np.array([r['nmse'] for r in eval_results['details']
                      if np.isfinite(r.get('nmse', float('nan')))])
log_nmse  = np.log10(nmse_vals + 1e-20)
sorted_ln = np.sort(log_nmse)
cumulative = np.arange(1, len(sorted_ln) + 1) / len(sorted_ln) * 100

axes[1].plot(sorted_ln, cumulative, 'b-', linewidth=2, label='SymbolicGPT (ours)')
axes[1].set_xlabel('Log₁₀(Normalised MSE)')
axes[1].set_ylabel('Proportion (%)')
axes[1].set_title('Cumulative Log NMSE Distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig('r2_distribution.png', dpi=120)
plt.show()

## 13. Final Results Summary Table

In [ ]:
print('\n' + '='*70)
print('            FINAL RESULTS SUMMARY')
print('='*70)
print(f'  Architecture : SymbolicGPT (T-Net + GPT Decoder + BFGS)')
print(f'  Training data: {CFG["train_size"]} real Feynman/Bonus samples')
print(f'  Variables    : {CFG["n_vars"]}  |  Points per sample : {CFG["n_points"]}')
print(f'  Embedding dim: {CFG["emb_dim"]}  |  Layers: {CFG["n_layers"]}  '
      f'|  Heads: {CFG["n_heads"]}')
print()
print('  Validation-set metrics:')
print(f"    R² (mean)        : {eval_results['r2_mean']:.4f}")
print(f"    RMSE (mean)      : {eval_results['rmse_mean']:.4f}")
print(f"    Token Accuracy   : {eval_results['token_acc_mean']*100:.2f}%")
print(f"    Exact Match      : {eval_results['exact_match']*100:.2f}%")
print()
print('  Comparison (from NeurIPS ML4PS 2025, Feynman dataset):')
print(f'    AI Feynman (2020)  Exact: ~100%  R² ≈ 1.00  (heuristic graph)')
print(f'    LASR (2024)        Exact:  72%   R² ≈ 0.99  (concept library)')
print(f'    QDSR (2024)        Exact:  91.6% R² ≈ 0.99  (discrete search)')
print(f'    Malik et al. 2025  Exact:  19.6% R² ≈ 0.976 (neural baseline)')
print(f'    Ours (this run)    Exact: {eval_results["exact_match"]*100:5.1f}% '
      f'R² ≈ {eval_results["r2_mean"]:.3f}  (SymbolicGPT reimplementation)')
print('='*70)

## 14. Save & Load Model Checkpoint

In [ ]:
checkpoint_path = 'symbolic_gpt_checkpoint.pt'

cfg_to_save = dict(CFG)
cfg_to_save['max_vars'] = MAX_VARS

torch.save({
    'model_state_dict': model.state_dict(),
    'config':           cfg_to_save,
    'vocab':            VOCAB,
    'history':          history,
}, checkpoint_path)
print(f'Model saved to {checkpoint_path}')


def load_model(path: str):
    ckpt = torch.load(path, map_location=DEVICE)
    cfg  = ckpt['config']
    max_vars_ckpt = int(cfg.get('max_vars', 5))
    m    = SymbolicGPT(
        in_dim      = max_vars_ckpt + 1,
        vocab_size  = VOCAB_SIZE,
        emb_dim     = cfg['emb_dim'],
        n_heads     = cfg['n_heads'],
        n_layers    = cfg['n_layers'],
        max_seq_len = cfg['max_seq_len'],
        dropout     = cfg['dropout'],
    ).to(DEVICE)
    m.load_state_dict(ckpt['model_state_dict'])
    m.eval()
    return m


loaded_model = load_model(checkpoint_path)
print('Model reloaded successfully.')

## 15. Interactive Inference

Given a new point cloud `(X, y)`, run inference to discover the equation.

In [ ]:
def tokens_to_sympy(tokens: list[str]) -> sp.Expr | None:
    """Convert prefix tokens to a SymPy expression via infix reconstruction."""
    infix = prefix_tokens_to_infix(tokens)
    local_syms = {f'x{i}': sp.Symbol(f'x{i}') for i in range(1, 6)}
    try:
        return sp.sympify(infix, locals=local_syms)
    except Exception:
        return None


EMPTY_PARAMS = np.array([])


def _pad_X_to_max_vars(X: np.ndarray, max_vars: int = MAX_VARS) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    X_pad = np.zeros((X.shape[0], max_vars), dtype=np.float32)
    X_pad[:, :min(X.shape[1], max_vars)] = X[:, :min(X.shape[1], max_vars)]
    return X_pad


def optimize_constants(equation, X: np.ndarray, y: np.ndarray, n_vars: int, n_restarts: int = 5) -> str:
    """Fit <C> placeholders with BFGS and return the final equation string."""
    skeleton = str(equation) if equation is not None else ''
    if not skeleton:
        return ''
    if '<C>' not in skeleton:
        return skeleton

    consts, _ = bfgs_fit_constants(skeleton, X, y, n_vars=n_vars, n_restarts=n_restarts)
    return fill_skeleton(skeleton, consts)


def compute_mse(eq, X: np.ndarray, y: np.ndarray, n_vars: int) -> float:
    """Compute equation MSE against targets; return inf on invalid evaluation."""
    try:
        eq_str = str(eq)
        eval_fn, _ = skeleton_to_callable(eq_str, n_vars)
        if eval_fn is None:
            return float('inf')
        y_pred = eval_fn(X, EMPTY_PARAMS)
        return float(np.mean((y_pred - y) ** 2))
    except Exception:
        return float('inf')


OPERATORS = [op for op in ['add', 'mul', 'sub', 'div'] if op in VOCAB]
MUTATION_PROBABILITY = 0.3


def mutate(eq_tokens: list[str]) -> list[str]:
    """Operator-aware mutation for hybrid neural-symbolic candidate refinement."""
    new_tokens = list(eq_tokens)

    for i, token in enumerate(new_tokens):
        if token in OPERATORS:
            if random.random() < MUTATION_PROBABILITY:
                new_tokens[i] = random.choice(OPERATORS)

    return new_tokens


def baseline_model(X: np.ndarray, y: np.ndarray) -> float:
    """Simple linear baseline (R² on training points)."""
    if HAS_SKLEARN:
        model = LinearRegression()
        model.fit(X, y)
        return float(model.score(X, y))

    X_aug = np.concatenate([X, np.ones((len(X), 1), dtype=X.dtype)], axis=1)
    coef, *_ = np.linalg.lstsq(X_aug, y, rcond=None)
    y_pred = X_aug @ coef
    return r2_score(y, y_pred)


def generate_candidates(model, cloud_t: torch.Tensor, num_samples: int = 5,
                        temperature: float = 0.7, top_k: int = 40) -> list[list[str]]:
    """Generate token candidates via greedy + top-k + mutation."""
    if num_samples < 1:
        num_samples = 1

    candidates = []
    safe_top_k = max(1, min(top_k, VOCAB_SIZE - 1))

    greedy_ids = model.generate(cloud_t, temperature=1.0, top_k=1)
    candidates.append(mutate(filter_special_tokens(ids_to_tokens(greedy_ids))))

    for _ in range(max(0, num_samples - 1)):
        eq_ids = model.generate(cloud_t, temperature=temperature, top_k=safe_top_k)
        candidates.append(mutate(filter_special_tokens(ids_to_tokens(eq_ids))))

    return candidates


def select_best(candidates: list[str], X: np.ndarray, y: np.ndarray, n_vars: int) -> tuple[str | None, float]:
    """Select the candidate equation with minimum MSE on (X, y)."""
    best = None
    best_loss = float('inf')

    for eq in candidates:
        loss = compute_mse(eq, X, y, n_vars=n_vars)
        if loss < best_loss:
            best = eq
            best_loss = loss

    return best, best_loss


def full_inference_pipeline(model, X: np.ndarray, y: np.ndarray,
                            n_vars: int, num_samples: int = 5,
                            temperature: float = 0.7, top_k: int = 40,
                            n_bfgs_restarts: int = 5) -> dict:
    """Run generation + selection + BFGS and return a structured result dict."""
    X_pad = _pad_X_to_max_vars(X, MAX_VARS)
    cloud_np = np.concatenate([X_pad, y.reshape(-1, 1)], axis=1).astype(np.float32)
    cloud_t  = torch.tensor(cloud_np[np.newaxis], dtype=torch.float32, device=DEVICE)

    candidates = generate_candidates(
        model, cloud_t, num_samples=num_samples,
        temperature=temperature, top_k=top_k,
    )

    candidate_eqs = []
    for toks in candidates:
        equation = tokens_to_sympy(toks)
        candidate_eqs.append(str(equation) if equation is not None else prefix_tokens_to_infix(toks))

    best_eq, best_mse = select_best(candidate_eqs, X_pad, y, n_vars=MAX_VARS)
    final_eq = optimize_constants(best_eq, X_pad, y, n_vars=MAX_VARS, n_restarts=n_bfgs_restarts)

    print(f'Best pre-BFGS candidate MSE: {best_mse:.6f}')
    return {
        'candidates': candidate_eqs,
        'best_pre_bfgs': best_eq,
        'best_pre_bfgs_mse': best_mse,
        'final_equation': final_eq,
    }


def symbolic_regression_infer(X: np.ndarray,
                               y: np.ndarray,
                               model,
                               n_vars: int,
                               temperature: float = 0.7,
                               top_k: int = 40,
                               n_bfgs_restarts: int = 5,
                               num_samples: int = 5) -> str:
    """
    Full inference pipeline:
      tokens → symbolic expression → optimisation → final equation
    """
    result = full_inference_pipeline(
        model, X, y,
        n_vars=n_vars,
        num_samples=num_samples,
        temperature=temperature,
        top_k=top_k,
        n_bfgs_restarts=n_bfgs_restarts,
    )
    print(f"Final equation: {result['final_equation']}")
    return result['final_equation']


print('\n── End-to-end demo ───────────────────────────────────────────────')
X_demo = np.random.uniform(-3, 3, (200, CFG['n_vars'])).astype(np.float32)
y_demo = (np.sin(X_demo[:, 0]) + np.cos(X_demo[:, 1])).astype(np.float32)
X_demo_pad = _pad_X_to_max_vars(X_demo, MAX_VARS)

print('Input dataset sample (first 5 rows):')
print(np.round(np.concatenate([X_demo[:5], y_demo[:5, None]], axis=1), 4))

demo_result = full_inference_pipeline(
    model, X_demo, y_demo,
    n_vars=CFG['n_vars'],
    num_samples=5,
    temperature=0.7,
    top_k=40,
    n_bfgs_restarts=5,
)

pred_pre = demo_result['best_pre_bfgs']
pred_final = demo_result['final_equation']
print(f"\nPredicted equation (pre-BFGS): {pred_pre}")
print(f"Predicted equation (post-BFGS): {pred_final}")

for label, eq_str in [('Pre-BFGS', pred_pre), ('Post-BFGS', pred_final)]:
    eval_fn, _ = skeleton_to_callable(eq_str, MAX_VARS)
    if eval_fn is not None:
        y_hat = eval_fn(X_demo_pad, EMPTY_PARAMS)
        print(f"{label:<10s} | R²={r2_score(y_demo, y_hat):.4f}  RMSE={rmse(y_demo, y_hat):.4f}")

baseline_r2 = baseline_model(X_demo, y_demo)
print(f"Linear baseline | R²={baseline_r2:.4f}")
print('Ground truth    : sin(x1) + cos(x2)')


def evaluate_equation_metrics(eq_str: str, X: np.ndarray, y: np.ndarray, n_vars: int) -> tuple[float, float]:
    """Return (mse, r2) for an equation string, or (inf, -inf) if invalid."""
    try:
        eval_fn, _ = skeleton_to_callable(eq_str, n_vars)
        if eval_fn is None:
            return float('inf'), float('-inf')
        y_hat = eval_fn(X, EMPTY_PARAMS)
        mse = float(np.mean((y - y_hat) ** 2))
        return mse, r2_score(y, y_hat)
    except Exception:
        return float('inf'), float('-inf')


cloud_demo = np.concatenate([X_demo_pad, y_demo.reshape(-1, 1)], axis=1).astype(np.float32)
cloud_demo_t = torch.tensor(cloud_demo[np.newaxis], dtype=torch.float32, device=DEVICE)

transformer_tokens = filter_special_tokens(ids_to_tokens(model.generate(cloud_demo_t, temperature=1.0, top_k=1)))
transformer_expr = tokens_to_sympy(transformer_tokens)
transformer_eq = str(transformer_expr) if transformer_expr is not None else prefix_tokens_to_infix(transformer_tokens)
transformer_bfgs_eq = optimize_constants(transformer_eq, X_demo_pad, y_demo, n_vars=MAX_VARS, n_restarts=5)

hybrid_eq = pred_final

mse1, r2_1 = evaluate_equation_metrics(transformer_eq, X_demo_pad, y_demo, MAX_VARS)
mse2, r2_2 = evaluate_equation_metrics(transformer_bfgs_eq, X_demo_pad, y_demo, MAX_VARS)
mse3, r2_3 = evaluate_equation_metrics(hybrid_eq, X_demo_pad, y_demo, MAX_VARS)

results = pd.DataFrame({
    'Method': ['Transformer', 'Transformer + BFGS', 'Hybrid'],
    'MSE': [mse1, mse2, mse3],
    'R2': [r2_1, r2_2, r2_3],
})

print('\nPerformance comparison:')
print(results)

fig, ax = plt.subplots(figsize=(8, 4))
idx = np.arange(min(100, len(y_demo)))
ax.plot(idx, y_demo[:len(idx)], label='Ground truth', linewidth=2)

for label, eq_str, style in [
    ('Transformer', transformer_eq, '--'),
    ('Transformer + BFGS', transformer_bfgs_eq, '-.'),
    ('Hybrid', hybrid_eq, ':'),
]:
    eval_fn, _ = skeleton_to_callable(eq_str, MAX_VARS)
    if eval_fn is not None:
        y_hat = eval_fn(X_demo_pad, EMPTY_PARAMS)
        ax.plot(idx, y_hat[:len(idx)], style, label=label)

ax.set_title('Method Comparison: Ground Truth vs Predictions')
ax.set_xlabel('Sample index')
ax.set_ylabel('y')
ax.legend()
plt.tight_layout()
plt.show()

## 16. Architecture Diagram

The figure below summarises the full SymbolicGPT pipeline.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.axis('off')

boxes = [
    (0.04, 0.3, 0.14, 0.4, 'Point Cloud\n{(x,y)}ᵢ\n[N × (d+1)]',  'lightblue'),
    (0.23, 0.2, 0.14, 0.6, 'T-Net\n(Order-invariant\nEncoder)',      'lightyellow'),
    (0.42, 0.3, 0.14, 0.4, 'Embedding\nw_D ∈ ℝᵉ',                  'lightgreen'),
    (0.60, 0.1, 0.16, 0.8, 'GPT Decoder\n(8 Transformer\nBlocks)',   'lightsalmon'),
    (0.81, 0.2, 0.14, 0.6, 'Skeleton\ne.g. sin(<C>*x1)+<C>',        'plum'),
]

for x, y, w, h, label, color in boxes:
    rect = matplotlib.patches.FancyBboxPatch(
        (x, y), w, h,
        boxstyle='round,pad=0.02',
        facecolor=color, edgecolor='gray', linewidth=1.5,
        transform=ax.transAxes
    )
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, label,
            ha='center', va='center', fontsize=8, transform=ax.transAxes)

# Arrows
arrow_props = dict(arrowstyle='->', lw=1.5, color='black')
for x_start, x_end in [(0.18, 0.23), (0.37, 0.42), (0.56, 0.60), (0.76, 0.81)]:
    ax.annotate('', xy=(x_end, 0.5), xytext=(x_start, 0.5),
                xycoords='axes fraction', arrowprops=arrow_props)

# BFGS box below skeleton
rect2 = matplotlib.patches.FancyBboxPatch(
    (0.81, -0.12), 0.14, 0.28,
    boxstyle='round,pad=0.02',
    facecolor='wheat', edgecolor='gray', linewidth=1.5,
    transform=ax.transAxes
)
ax.add_patch(rect2)
ax.text(0.88, 0.04, 'BFGS\nConstant\nOptimisation',
        ha='center', va='center', fontsize=8, transform=ax.transAxes)
ax.annotate('', xy=(0.88, 0.2), xytext=(0.88, -0.02),
            xycoords='axes fraction',
            arrowprops=dict(arrowstyle='<->', lw=1.5, color='darkblue'))

ax.set_title('SymbolicGPT Architecture', fontsize=13, pad=10)
plt.tight_layout()
plt.savefig('architecture_diagram.png', dpi=120)
plt.show()

## 17. Conclusions

This notebook implements **SymbolicGPT** for symbolic regression, combining:

| Component | Role |
|---|---|
| **T-Net** | Order-invariant encoder of input point clouds |
| **GPT Decoder** | Auto-regressive equation skeleton generator |
| **`<C>` masking** | Decouples structural learning from constant fitting |
| **BFGS optimisation** | Post-hoc constant refinement |

### Failure Cases

Observed issues:
- Correct structure but wrong constants
- Missing terms
- Incorrect exponent placement

Insight:
Model achieves strong numerical accuracy but struggles with exact symbolic recovery.

### Key Insights

- Neural models capture local structure effectively
- Global symbolic correctness remains challenging
- Separating structure learning and constant optimization improves performance

### Final Pipeline

Dataset → Tokenization → T-Net → Transformer → Skeleton Equation → BFGS → Hybrid Refinement → Final Equation

### Future directions

* Integrate **PIGP / SymbolicDPO** (Jha et al., 2024) to refine skeletons with GP.
* Add **learned concept libraries** (Malik et al., 2025) for recurring sub-expressions.
* Scale to the full **AI Feynman** benchmark (100 equations, multi-variable).
* Apply **sparse attention** (sliding window) for longer equation sequences.
* Explore **curriculum learning** – start simple, grow complexity.
